In [1]:
123

123

In [2]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [22]:
import pandas as pd

train_df = pd.read_csv("../data/long_basic/train.csv")
val_df = pd.read_csv("../data/long_basic/validation.csv")
test_df = pd.read_csv("../data/long_basic/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (22430, 22)
Validation: (923, 22)
Test: (926, 22)


In [24]:
# 3. Device 설정
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [3]:
target_col = "next_goals"

In [4]:
basic_numeric_cols = [
    "age",
    "starts",
    "minutes",
    "goals",
    "assists",
    "non_penalty_goals",
    "penalty_goals",
    "penalty_attempts"
]

In [5]:
per90_cols = [
    "goals_per90",
    "assists_per90",
    "goal_contrib_per90"
]

In [6]:
context_cols = [
    "league",
    "position_group"
]

In [7]:
team_col = ["team"]

In [8]:
experiments = {
    "Exp0_full_baseline": {
        "numeric": basic_numeric_cols + per90_cols,
        "categorical": context_cols
    },

    "Exp1_basic_only": {
        "numeric": basic_numeric_cols,
        "categorical": []
    },

    "Exp2_basic_per90": {
        "numeric": basic_numeric_cols + per90_cols,
        "categorical": []
    },

    "Exp3_basic_per90_context": {
        "numeric": basic_numeric_cols + per90_cols,
        "categorical": context_cols
    }
}

In [9]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [10]:
def prepare_data(train_df, val_df, numeric_cols, categorical_cols):
    feature_cols = numeric_cols + categorical_cols

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]

    y_train = train_df["next_goals"]
    y_val = val_df["next_goals"]

    transformers = []

    if numeric_cols:
        transformers.append(
            ("num", StandardScaler(), numeric_cols)
        )

    if categorical_cols:
        transformers.append(
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_cols
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers
    )

    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)

    return (
        X_train_processed,
        X_val_processed,
        y_train,
        y_val,
        preprocessor
    )

In [11]:
from torch.utils.data import TensorDataset, DataLoader

In [12]:
def make_loaders(
    X_train_processed,
    X_val_processed,
    y_train,
    y_val,
    batch_size=64
):
    X_train_tensor = torch.tensor(
        X_train_processed,
        dtype=torch.float32
    )

    X_val_tensor = torch.tensor(
        X_val_processed,
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train.values,
        dtype=torch.float32
    ).reshape(-1, 1)

    y_val_tensor = torch.tensor(
        y_val.values,
        dtype=torch.float32
    ).reshape(-1, 1)

    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor
    )

    val_dataset = TensorDataset(
        X_val_tensor,
        y_val_tensor
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader

In [15]:
import torch
import torch.nn as nn

class MLPRegression(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [16]:
def train_model(
    model,
    train_loader,
    val_loader,
    epochs=50,
    lr=0.001
):
    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # Train
        model.train()

        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            pred = model(X_batch)
            loss = criterion(pred, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)


        # Validation
        model.eval()

        val_loss = 0.0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                pred = model(X_batch)
                loss = criterion(pred, y_batch)

                val_loss += loss.item()

        val_loss /= len(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

    return model, train_losses, val_losses

In [99]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def run_experiment(
    experiment_name,
    train_df,
    val_df,
    numeric_cols,
    categorical_cols,
    epochs=50,
    batch_size=64,
    lr=0.001
):
    # -------------------------
    # 1. Seed 고정
    # -------------------------
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


    # -------------------------
    # 2. 전처리
    # -------------------------
    (
        X_train_processed,
        X_val_processed,
        y_train,
        y_val,
        preprocessor
    ) = prepare_data(
        train_df,
        val_df,
        numeric_cols,
        categorical_cols
    )


    # -------------------------
    # 3. DataLoader 생성
    # -------------------------
    train_loader, val_loader = make_loaders(
        X_train_processed,
        X_val_processed,
        y_train,
        y_val,
        batch_size=batch_size
    )


    # -------------------------
    # 4. 모델 생성
    # -------------------------
    input_dim = X_train_processed.shape[1]

    model = MLPRegression(
        input_dim=input_dim
    ).to(device)


    # -------------------------
    # 5. 모델 학습
    # -------------------------
    model, train_losses, val_losses = train_model(
        model,
        train_loader,
        val_loader,
        epochs=epochs,
        lr=lr
    )


    # -------------------------
    # 6. Validation 예측
    # -------------------------
    model.eval()

    val_preds = []
    val_targets = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)

            pred = model(X_batch)

            val_preds.extend(
                pred.cpu().numpy().flatten()
            )

            val_targets.extend(
                y_batch.numpy().flatten()
            )


    # -------------------------
    # 7. 평가 지표 계산
    # -------------------------
    mae = mean_absolute_error(
        val_targets,
        val_preds
    )

    rmse = mean_squared_error(
        val_targets,
        val_preds
    ) ** 0.5

    r2 = r2_score(
        val_targets,
        val_preds
    )


    # -------------------------
    # 8. 결과 출력
    # -------------------------
    print(f"\n[{experiment_name}]")
    print(f"Input Dimension : {input_dim}")
    print(f"Validation MAE  : {mae:.4f}")
    print(f"Validation RMSE : {rmse:.4f}")
    print(f"Validation R²   : {r2:.4f}")


    # -------------------------
    # 9. 결과 반환
    # -------------------------
    return {
    "experiment": experiment_name,
    "input_dim": input_dim,
    "mae": mae,
    "rmse": rmse,
    "r2": r2,
    "train_loss": train_losses[-1],
    "val_loss": val_losses[-1],

    # 추가
    "val_preds": np.array(val_preds),
    "val_targets": np.array(val_targets)
    }

In [25]:

exp = experiments["Exp1_basic_only"]

result_exp1 = run_experiment(
    experiment_name="Exp1_basic_only",
    train_df=train_df,
    val_df=val_df,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp1_basic_only]
Input Dimension : 8
Validation MAE  : 2.4262
Validation RMSE : 3.5181
Validation R²   : 0.4076


In [26]:
results = []

# Exp1
results.append(result_exp1)


# Exp2: basic + per90
exp = experiments["Exp2_basic_per90"]

result_exp2 = run_experiment(
    experiment_name="Exp2_basic_per90",
    train_df=train_df,
    val_df=val_df,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)

results.append(result_exp2)


# Exp3: basic + per90 + league + position
exp = experiments["Exp3_basic_per90_context"]

result_exp3 = run_experiment(
    experiment_name="Exp3_basic_per90_context",
    train_df=train_df,
    val_df=val_df,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)

results.append(result_exp3)


[Exp2_basic_per90]
Input Dimension : 11
Validation MAE  : 2.4105
Validation RMSE : 3.5210
Validation R²   : 0.4066

[Exp3_basic_per90_context]
Input Dimension : 18
Validation MAE  : 2.4037
Validation RMSE : 3.4829
Validation R²   : 0.4193


In [27]:
results_df = pd.DataFrame(results)

results_df[
    ["experiment", "input_dim", "mae", "rmse", "r2"]
]

,experiment,input_dim,mae,rmse,r2
0,Exp1_basic_only,8,2.426160,3.518096,0.407554
1,Exp2_basic_per90,11,2.410517,3.521004,0.406574
2,Exp3_basic_per90_context,18,2.403708,3.482921,0.419342


### 피처 그룹 실험 결과

기본 기록만 사용한 Exp1의 Validation MAE는 2.426이었다.

90분당 기록을 추가한 Exp2에서는 MAE가 2.411로 소폭 개선되었으나,
RMSE와 R²는 거의 개선되지 않았다.
따라서 per90 피처의 추가 효과는 제한적인 것으로 나타났다.

리그와 포지션 정보를 추가한 Exp3에서는
MAE 2.404, RMSE 3.483, R² 0.419로 세 지표가 모두 개선되었다.

따라서 현재 실험에서는 선수의 단순 경기 기록뿐 아니라
리그와 포지션 같은 맥락 정보가 다음 시즌 득점 예측에 유의미한
추가 정보를 제공하는 것으로 판단된다.

In [29]:
# Exp4: 나이 피처부터 시작
def add_age_features(df):
    df = df.copy()

    df["age_squared"] = df["age"] ** 2

    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 20, 23, 27, 30, 100],
        labels=[
            "U20",
            "21_23",
            "24_27",
            "28_30",
            "31_plus"
        ]
    )

    return df

In [31]:
train_fe = add_age_features(train_df)
val_fe = add_age_features(val_df)
test_fe = add_age_features(test_df)

In [32]:
print(
    train_fe[
        ["age", "age_squared", "age_group"]
    ].head(10)
)

print(train_fe["age_group"].value_counts().sort_index())

    age  age_squared age_group
0  31.0        961.0   31_plus
1  26.0        676.0     24_27
2  23.0        529.0     21_23
3  25.0        625.0     24_27
4  27.0        729.0     24_27
5  21.0        441.0     21_23
6  22.0        484.0     21_23
7  24.0        576.0     24_27
8  27.0        729.0     24_27
9  22.0        484.0     21_23
age_group
U20        1794
21_23      4987
24_27      8312
28_30      4581
31_plus    2756
Name: count, dtype: int64


In [33]:
experiments["Exp4_add_age_features"] = {
    "numeric": basic_numeric_cols + per90_cols + ["age_squared"],
    "categorical": context_cols + ["age_group"]
}

In [34]:
exp = experiments["Exp4_add_age_features"]

result_exp4 = run_experiment(
    experiment_name="Exp4_add_age_features",
    train_df=train_fe,
    val_df=val_fe,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)

print(result_exp4)


[Exp4_add_age_features]
Input Dimension : 24
Validation MAE  : 2.4440
Validation RMSE : 3.5026
Validation R²   : 0.4127
{'experiment': 'Exp4_add_age_features', 'input_dim': 24, 'mae': 2.4440099580348638, 'rmse': 3.5026481908406546, 'r2': 0.4127452940907431, 'train_loss': 10.619152181848161, 'val_loss': 12.42444626490275}


In [35]:
experiments["Exp4a_age_squared"] = {
    "numeric": basic_numeric_cols + per90_cols + ["age_squared"],
    "categorical": context_cols
}

In [36]:
exp = experiments["Exp4a_age_squared"]

result_exp4a = run_experiment(
    experiment_name="Exp4a_age_squared",
    train_df=train_fe,
    val_df=val_fe,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp4a_age_squared]
Input Dimension : 19
Validation MAE  : 2.4157
Validation RMSE : 3.5216
Validation R²   : 0.4064


In [37]:
experiments["Exp4b_age_group"] = {
    "numeric": basic_numeric_cols + per90_cols,
    "categorical": context_cols + ["age_group"]
}

In [38]:
experiments["Exp4b_age_group"] = {
    "numeric": basic_numeric_cols + per90_cols,
    "categorical": context_cols + ["age_group"]
}

In [39]:
exp = experiments["Exp4b_age_group"]

result_exp4b = run_experiment(
    experiment_name="Exp4b_age_group",
    train_df=train_fe,
    val_df=val_fe,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp4b_age_group]
Input Dimension : 23
Validation MAE  : 2.4348
Validation RMSE : 3.5297
Validation R²   : 0.4036


| 실험                    |        MAE |       RMSE |         R² | 판단 |
| --------------------- | ---------: | ---------: | ---------: | -- |
| Exp3 기본+per90+context | **2.4037** | **3.4829** | **0.4193** | 기준 |
| Exp4a + age²          |     2.4157 |     3.5216 |     0.4064 | 악화 |
| Exp4b + age_group     |     2.4348 |     3.5297 |     0.4036 | 악화 |
| Exp4 + 둘 다            |     2.4440 |     3.5026 |     0.4127 | 악화 |


In [40]:
def add_current_season_features(df):
    df = df.copy()

    # 출전 안정성
    df["minutes_per_start"] = (
        df["minutes"] / df["starts"].replace(0, np.nan)
    )

    # 총 공격 기여
    df["goal_contrib"] = (
        df["goals"] + df["assists"]
    )

    # 득점/도움 성향
    contrib = df["goal_contrib"].replace(0, np.nan)

    df["goal_share"] = (
        df["goals"] / contrib
    )

    df["assist_share"] = (
        df["assists"] / contrib
    )

    # PK 성공률
    df["penalty_conversion"] = (
        df["penalty_goals"]
        / df["penalty_attempts"].replace(0, np.nan)
    )

    # 비PK 득점 비율
    df["non_penalty_goal_ratio"] = (
        df["non_penalty_goals"]
        / df["goals"].replace(0, np.nan)
    )

    # 0으로 나누면서 생긴 NaN 처리
    new_cols = [
        "minutes_per_start",
        "goal_contrib",
        "goal_share",
        "assist_share",
        "penalty_conversion",
        "non_penalty_goal_ratio"
    ]

    df[new_cols] = df[new_cols].fillna(0)

    return df

In [41]:
train_fe2 = add_current_season_features(train_df)
val_fe2 = add_current_season_features(val_df)
test_fe2 = add_current_season_features(test_df)

In [42]:
current_feature_cols = [
    "minutes_per_start",
    "goal_contrib",
    "goal_share",
    "assist_share",
    "penalty_conversion",
    "non_penalty_goal_ratio"
]

print(train_fe2[current_feature_cols].describe().T)

                          count       mean       std      min        25%  \
minutes_per_start       22430.0  89.752029  9.109771  58.1875  85.114560   
goal_contrib            22430.0   6.462951  6.064792   0.0000   2.000000   
goal_share              22430.0   0.512711  0.330272   0.0000   0.272727   
assist_share            22430.0   0.408421  0.317575   0.0000   0.153846   
penalty_conversion      22430.0   0.150370  0.341371   0.0000   0.000000   
non_penalty_goal_ratio  22430.0   0.749230  0.400485   0.0000   0.666667   

                              50%        75%         max  
minutes_per_start       88.333333  92.000000  312.333333  
goal_contrib             5.000000   9.000000   66.000000  
goal_share               0.525063   0.758621    1.000000  
assist_share             0.375000   0.625000    1.000000  
penalty_conversion       0.000000   0.000000    1.000000  
non_penalty_goal_ratio   1.000000   1.000000    1.000000  


In [43]:
train_fe2[
    ["player", "team", "season", "starts", "minutes", "minutes_per_start"]
].sort_values(
    "minutes_per_start",
    ascending=False
).head(20)

,player,team,season,starts,minutes,minutes_per_start
5381,Roberto Pinto,Arminia,2005-2006,3.0,937.0,312.333333
11889,Fernando Llorente,Athletic Club,2012-2013,4.0,901.0,225.250000
16674,Stipe Perica,Udinese,2016-2017,5.0,1086.0,217.200000
3855,Duncan Ferguson,Everton,2004-2005,6.0,1196.0,199.333333
1795,Albert Luque,La Coruña,2002-2003,5.0,982.0,196.400000
14676,Souleymane Camara,Montpellier,2014-2015,7.0,1346.0,192.285714
15389,Manucho,Rayo Vallecano,2015-2016,5.0,949.0,189.800000
4197,Marco Borriello,Reggina,2004-2005,5.0,944.0,188.800000
15317,Kevin Lasagna,Carpi,2015-2016,8.0,1504.0,188.000000
15033,Dries Mertens,Napoli,2015-2016,6.0,1108.0,184.666667


In [44]:
experiments["Exp5a_goal_contrib"] = {
    "numeric": basic_numeric_cols
               + per90_cols
               + ["goal_contrib"],
    "categorical": context_cols
}

In [45]:
exp = experiments["Exp5a_goal_contrib"]

result_exp5a = run_experiment(
    experiment_name="Exp5a_goal_contrib",
    train_df=train_fe2,
    val_df=val_fe2,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp5a_goal_contrib]
Input Dimension : 19
Validation MAE  : 2.4166
Validation RMSE : 3.5277
Validation R²   : 0.4043


In [46]:
experiments["Exp5b_goal_share"] = {
    "numeric": basic_numeric_cols
               + per90_cols
               + ["goal_share"],
    "categorical": context_cols
}

In [47]:
def add_penalty_features(df):
    df = df.copy()

    df["penalty_taker"] = (
        df["penalty_attempts"] > 0
    ).astype(int)

    df["penalty_conversion"] = np.where(
        df["penalty_attempts"] > 0,
        df["penalty_goals"] / df["penalty_attempts"],
        0
    )

    return df

In [48]:
experiments["Exp5b_goal_share"] = {
    "numeric": basic_numeric_cols + per90_cols + ["goal_share"],
    "categorical": context_cols
}

exp = experiments["Exp5b_goal_share"]

result_exp5b = run_experiment(
    experiment_name="Exp5b_goal_share",
    train_df=train_fe2,
    val_df=val_fe2,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp5b_goal_share]
Input Dimension : 19
Validation MAE  : 2.4181
Validation RMSE : 3.5131
Validation R²   : 0.4092


In [49]:
def add_penalty_features(df):
    df = df.copy()

    df["penalty_taker"] = (
        df["penalty_attempts"] > 0
    ).astype(int)

    df["penalty_conversion"] = np.where(
        df["penalty_attempts"] > 0,
        df["penalty_goals"] / df["penalty_attempts"],
        0
    )

    return df

In [50]:
train_pk = add_penalty_features(train_df)
val_pk = add_penalty_features(val_df)
test_pk = add_penalty_features(test_df)

In [51]:
print(
    train_pk[
        [
            "penalty_goals",
            "penalty_attempts",
            "penalty_taker",
            "penalty_conversion"
        ]
    ].head(20)
)

    penalty_goals  penalty_attempts  penalty_taker  penalty_conversion
0             0.0               0.0              0            0.000000
1             0.0               0.0              0            0.000000
2             1.0               2.0              1            0.500000
3             0.0               0.0              0            0.000000
4             1.0               1.0              1            1.000000
5             0.0               1.0              1            0.000000
6             0.0               0.0              0            0.000000
7             0.0               0.0              0            0.000000
8             4.0               5.0              1            0.800000
9             0.0               0.0              0            0.000000
10            2.0               3.0              1            0.666667
11            0.0               0.0              0            0.000000
12            0.0               0.0              0            0.000000
13    

In [52]:
print(train_pk["penalty_taker"].value_counts())
print()
print(train_pk["penalty_conversion"].describe())

penalty_taker
0    17867
1     4563
Name: count, dtype: int64

count    22430.000000
mean         0.150370
std          0.341371
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: penalty_conversion, dtype: float64


In [53]:
experiments["Exp5c_penalty_features"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + ["penalty_taker", "penalty_conversion"]
    ),
    "categorical": context_cols
}

In [54]:
exp = experiments["Exp5c_penalty_features"]

result_exp5c = run_experiment(
    experiment_name="Exp5c_penalty_features",
    train_df=train_pk,
    val_df=val_pk,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp5c_penalty_features]
Input Dimension : 20
Validation MAE  : 2.4448
Validation RMSE : 3.5213
Validation R²   : 0.4065


### 현재 시즌 파생 피처 실험 결과

기존 피처를 바탕으로 나이 구간, 나이 제곱항, 공격 기여도,
득점 비중, 페널티 관련 파생 피처 등을 추가하여 검증하였다.

그러나 모든 실험에서 기준 모델인 Exp3
(MAE 2.404, RMSE 3.483, R² 0.419)보다 성능이 개선되지 않았다.

이는 기존 입력에 이미 goals, assists, penalty 관련 기록,
per90 지표 등이 포함되어 있어 단순 산술 조합으로 만든 파생 변수는
추가적인 정보를 충분히 제공하지 못했기 때문으로 해석할 수 있다.

따라서 이후 실험에서는 같은 시즌 기록의 재조합보다
직전 시즌 및 최근 여러 시즌의 선수 기록을 활용한 시계열적 피처를
추가하는 방향으로 진행한다.

In [55]:
train_temp = train_df.copy()
train_temp["split"] = "train"

val_temp = val_df.copy()
val_temp["split"] = "val"

test_temp = test_df.copy()
test_temp["split"] = "test"

full_df = pd.concat(
    [train_temp, val_temp, test_temp],
    ignore_index=True
)

print(full_df.shape)
print(full_df["split"].value_counts())

(24279, 23)
split
train    22430
test       926
val        923
Name: count, dtype: int64


In [56]:
def add_previous_season_features(df):
    df = df.copy()

    # 선수별 시간순 정렬
    df = df.sort_values(
        ["player", "season"]
    ).copy()

    # 직전 기록
    df["prev_goals"] = (
        df.groupby("player")["goals"]
        .shift(1)
    )

    df["prev_minutes"] = (
        df.groupby("player")["minutes"]
        .shift(1)
    )

    df["prev_goals_per90"] = (
        df.groupby("player")["goals_per90"]
        .shift(1)
    )

    return df

In [57]:
full_prev = add_previous_season_features(full_df)

In [58]:
train_prev = full_prev[
    full_prev["split"] == "train"
].copy()

val_prev = full_prev[
    full_prev["split"] == "val"
].copy()

test_prev = full_prev[
    full_prev["split"] == "test"
].copy()

In [59]:
full_prev[
    full_prev["player"] == "Harry Kane"
][
    [
        "player",
        "season",
        "goals",
        "prev_goals",
        "goals_per90",
        "prev_goals_per90"
    ]
]

,player,season,goals,prev_goals,goals_per90,prev_goals_per90
14111,Harry Kane,2014-2015,21.0,NaN,0.732274,NaN
15162,Harry Kane,2015-2016,25.0,21.0,0.669444,0.732274
16146,Harry Kane,2016-2017,29.0,25.0,1.035303,0.669444
17118,Harry Kane,2017-2018,30.0,29.0,0.877763,1.035303
18061,Harry Kane,2018-2019,17.0,30.0,0.631188,0.877763
19008,Harry Kane,2019-2020,18.0,17.0,0.626208,0.631188
19922,Harry Kane,2020-2021,23.0,18.0,0.671642,0.626208
20874,Harry Kane,2021-2022,17.0,23.0,0.473391,0.671642
21833,Harry Kane,2022-2023,30.0,17.0,0.792952,0.473391
22760,Harry Kane,2023-2024,36.0,30.0,1.141247,0.792952


In [60]:
def add_previous_season_features(df):
    df = df.copy()

    # 2014-2015 -> 2014
    df["season_start"] = (
        df["season"]
        .str[:4]
        .astype(int)
    )

    df = df.sort_values(
        ["player", "season_start"]
    ).copy()

    grouped = df.groupby("player")

    # 이전에 관측된 시즌
    df["prev_season_start"] = (
        grouped["season_start"]
        .shift(1)
    )

    # 이전 행의 기록
    df["prev_goals"] = (
        grouped["goals"]
        .shift(1)
    )

    df["prev_minutes"] = (
        grouped["minutes"]
        .shift(1)
    )

    df["prev_goals_per90"] = (
        grouped["goals_per90"]
        .shift(1)
    )

    # 정말 바로 전 시즌인가?
    df["has_prev_season"] = (
        df["season_start"] - df["prev_season_start"] == 1
    ).astype(int)

    # 시즌이 연속되지 않았다면 직전 시즌 기록으로 인정하지 않음
    prev_cols = [
        "prev_goals",
        "prev_minutes",
        "prev_goals_per90"
    ]

    df.loc[
        df["has_prev_season"] == 0,
        prev_cols
    ] = np.nan

    return df

In [61]:
full_prev = add_previous_season_features(full_df)

train_prev = full_prev[
    full_prev["split"] == "train"
].copy()

val_prev = full_prev[
    full_prev["split"] == "val"
].copy()

test_prev = full_prev[
    full_prev["split"] == "test"
].copy()

In [62]:
prev_cols = [
    "prev_goals",
    "prev_minutes",
    "prev_goals_per90"
]

train_prev[prev_cols] = train_prev[prev_cols].fillna(0)
val_prev[prev_cols] = val_prev[prev_cols].fillna(0)
test_prev[prev_cols] = test_prev[prev_cols].fillna(0)

In [63]:
experiments["Exp6_previous_season"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "prev_goals",
            "prev_minutes",
            "prev_goals_per90",
            "has_prev_season"
        ]
    ),
    "categorical": context_cols
}

In [64]:
exp = experiments["Exp6_previous_season"]

result_exp6 = run_experiment(
    experiment_name="Exp6_previous_season",
    train_df=train_prev,
    val_df=val_prev,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp6_previous_season]
Input Dimension : 22
Validation MAE  : 2.3887
Validation RMSE : 3.5443
Validation R²   : 0.3987


In [65]:
print(train_prev["has_prev_season"].value_counts())
print()
print(val_prev["has_prev_season"].value_counts())

has_prev_season
1    13912
0     8518
Name: count, dtype: int64

has_prev_season
1    559
0    364
Name: count, dtype: int64


In [66]:
experiments["Exp6a_prev_goals"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + ["prev_goals", "has_prev_season"]
    ),
    "categorical": context_cols
}

In [67]:
experiments["Exp6b_prev_goals_per90"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + ["prev_goals_per90", "has_prev_season"]
    ),
    "categorical": context_cols
}

In [68]:
experiments["Exp6c_prev_scoring"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "prev_goals",
            "prev_goals_per90",
            "has_prev_season"
        ]
    ),
    "categorical": context_cols
}

In [69]:
exp = experiments["Exp6a_prev_goals"]

result_exp6a = run_experiment(
    experiment_name="Exp6a_prev_goals",
    train_df=train_prev,
    val_df=val_prev,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp6a_prev_goals]
Input Dimension : 20
Validation MAE  : 2.3993
Validation RMSE : 3.5336
Validation R²   : 0.4023


In [70]:
exp = experiments["Exp6b_prev_goals_per90"]

result_exp6b = run_experiment(
    experiment_name="Exp6b_prev_goals_per90",
    train_df=train_prev,
    val_df=val_prev,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp6b_prev_goals_per90]
Input Dimension : 20
Validation MAE  : 2.4134
Validation RMSE : 3.5378
Validation R²   : 0.4009


In [71]:
exp = experiments["Exp6c_prev_scoring"]

result_exp6c = run_experiment(
    experiment_name="Exp6c_prev_scoring",
    train_df=train_prev,
    val_df=val_prev,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp6c_prev_scoring]
Input Dimension : 21
Validation MAE  : 2.3654
Validation RMSE : 3.5321
Validation R²   : 0.4028


In [72]:
def add_rolling_features(df):
    df = df.copy()

    df["season_start"] = (
        df["season"]
        .str[:4]
        .astype(int)
    )

    df = df.sort_values(
        ["player", "season_start"]
    ).copy()

    grouped = df.groupby("player")

    # 현재 시즌을 제외한 과거 기록
    prev_goals = grouped["goals"].shift(1)
    prev_goals_per90 = grouped["goals_per90"].shift(1)
    prev_minutes = grouped["minutes"].shift(1)

    # 최근 최대 3시즌 평균
    df["goals_3yr_mean"] = (
        prev_goals
        .groupby(df["player"])
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["goals_per90_3yr_mean"] = (
        prev_goals_per90
        .groupby(df["player"])
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["minutes_3yr_mean"] = (
        prev_minutes
        .groupby(df["player"])
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # 최근 최대 3시즌 최고 득점
    df["goals_3yr_max"] = (
        prev_goals
        .groupby(df["player"])
        .rolling(window=3, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )

    # 현재 시즌 이전까지 등장한 시즌 수
    df["career_seasons"] = grouped.cumcount()

    return df

In [73]:
full_roll = add_rolling_features(full_df)

train_roll = full_roll[
    full_roll["split"] == "train"
].copy()

val_roll = full_roll[
    full_roll["split"] == "val"
].copy()

test_roll = full_roll[
    full_roll["split"] == "test"
].copy()

In [74]:
full_roll[
    full_roll["player"] == "Harry Kane"
][
    [
        "season",
        "goals",
        "goals_3yr_mean",
        "goals_per90_3yr_mean",
        "minutes_3yr_mean",
        "goals_3yr_max",
        "career_seasons"
    ]
]

,season,goals,goals_3yr_mean,goals_per90_3yr_mean,minutes_3yr_mean,goals_3yr_max,career_seasons
14111,2014-2015,21.0,NaN,NaN,NaN,NaN,0
15162,2015-2016,25.0,21.000000,0.732274,2581.000000,21.0,1
16146,2016-2017,29.0,23.000000,0.700859,2971.000000,25.0,2
17118,2017-2018,30.0,25.000000,0.812340,2821.000000,29.0,3
18061,2018-2019,17.0,28.000000,0.860837,2986.000000,30.0,4
19008,2019-2020,18.0,25.333333,0.848085,2673.666667,30.0,5
19922,2020-2021,23.0,21.666667,0.711720,2695.666667,30.0,6
20874,2021-2022,17.0,19.333333,0.643013,2697.666667,23.0,7
21833,2022-2023,30.0,19.333333,0.590414,2967.000000,23.0,8
22760,2023-2024,36.0,23.333333,0.645995,3239.666667,30.0,9


In [76]:
rolling_cols = [
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "minutes_3yr_mean",
    "goals_3yr_max"
]

train_roll[rolling_cols] = (
    train_roll[rolling_cols].fillna(0)
)

val_roll[rolling_cols] = (
    val_roll[rolling_cols].fillna(0)
)

test_roll[rolling_cols] = (
    test_roll[rolling_cols].fillna(0)
)

In [77]:
experiments["Exp7_rolling_3yr"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "goals_3yr_mean",
            "goals_per90_3yr_mean",
            "minutes_3yr_mean",
            "goals_3yr_max",
            "career_seasons"
        ]
    ),
    "categorical": context_cols
}

In [78]:
exp = experiments["Exp7_rolling_3yr"]

result_exp7 = run_experiment(
    experiment_name="Exp7_rolling_3yr",
    train_df=train_roll,
    val_df=val_roll,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp7_rolling_3yr]
Input Dimension : 23
Validation MAE  : 2.3792
Validation RMSE : 3.5216
Validation R²   : 0.4064


In [79]:
experiments["Exp7a_scoring_trend"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "goals_3yr_mean",
            "goals_per90_3yr_mean",
            "goals_3yr_max"
        ]
    ),
    "categorical": context_cols
}

experiments["Exp7b_experience_stability"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "minutes_3yr_mean",
            "career_seasons"
        ]
    ),
    "categorical": context_cols
}

experiments["Exp7c_prev_plus_trend"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "prev_goals",
            "prev_goals_per90",
            "has_prev_season",
            "goals_3yr_mean",
            "goals_per90_3yr_mean",
            "goals_3yr_max"
        ]
    ),
    "categorical": context_cols
}

In [80]:
full_combined = add_previous_season_features(full_df)
full_combined = add_rolling_features(full_combined)

train_combined = full_combined[
    full_combined["split"] == "train"
].copy()

val_combined = full_combined[
    full_combined["split"] == "val"
].copy()

test_combined = full_combined[
    full_combined["split"] == "test"
].copy()

In [81]:
prev_cols = [
    "prev_goals",
    "prev_minutes",
    "prev_goals_per90"
]

rolling_cols = [
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "minutes_3yr_mean",
    "goals_3yr_max"
]

fill_cols = prev_cols + rolling_cols

train_combined[fill_cols] = train_combined[fill_cols].fillna(0)
val_combined[fill_cols] = val_combined[fill_cols].fillna(0)
test_combined[fill_cols] = test_combined[fill_cols].fillna(0)

In [82]:
exp = experiments["Exp7a_scoring_trend"]

result_exp7a = run_experiment(
    experiment_name="Exp7a_scoring_trend",
    train_df=train_combined,
    val_df=val_combined,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp7a_scoring_trend]
Input Dimension : 21
Validation MAE  : 2.3689
Validation RMSE : 3.5470
Validation R²   : 0.3978


In [83]:
exp = experiments["Exp7b_experience_stability"]

result_exp7b = run_experiment(
    experiment_name="Exp7b_experience_stability",
    train_df=train_combined,
    val_df=val_combined,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp7b_experience_stability]
Input Dimension : 20
Validation MAE  : 2.4404
Validation RMSE : 3.5305
Validation R²   : 0.4034


In [84]:
exp = experiments["Exp7c_prev_plus_trend"]

result_exp7c = run_experiment(
    experiment_name="Exp7c_prev_plus_trend",
    train_df=train_combined,
    val_df=val_combined,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp7c_prev_plus_trend]
Input Dimension : 24
Validation MAE  : 2.4191
Validation RMSE : 3.5394
Validation R²   : 0.4004


In [85]:
trend_results = pd.DataFrame([
    result_exp7a,
    result_exp7b,
    result_exp7c
])

trend_results[
    ["experiment", "input_dim", "mae", "rmse", "r2"]
]

,experiment,input_dim,mae,rmse,r2
0,Exp7a_scoring_trend,21,2.368913,3.547010,0.397776
1,Exp7b_experience_stability,20,2.440364,3.530524,0.403361
2,Exp7c_prev_plus_trend,24,2.419144,3.539396,0.400358


In [86]:
def add_teammate_features(df):
    df = df.copy()

    group_cols = ["season", "team"]

    source_cols = [
        "goals",
        "assists",
        "goals_per90",
        "goal_contrib_per90"
    ]

    grouped = df.groupby(group_cols)

    for col in source_cols:
        # 팀 전체 합계
        group_sum = grouped[col].transform("sum")

        # 팀 내 분석 대상 선수 수
        group_count = grouped[col].transform("count")

        # 자기 자신을 제외한 팀 동료 평균
        df[f"teammate_{col}_mean"] = np.where(
            group_count > 1,
            (group_sum - df[col]) / (group_count - 1),
            0
        )

    # 같은 조건을 만족한 공격 선수/미드필더 동료 수
    df["teammate_count"] = (
        grouped["player"].transform("count") - 1
    )

    return df

In [87]:
train_team = add_teammate_features(train_df)
val_team = add_teammate_features(val_df)
test_team = add_teammate_features(test_df)

In [88]:
teammate_cols = [
    "teammate_goals_mean",
    "teammate_assists_mean",
    "teammate_goals_per90_mean",
    "teammate_goal_contrib_per90_mean",
    "teammate_count"
]

print(
    train_team[teammate_cols]
    .describe()
    .T
)

                                    count      mean       std       min  \
teammate_goals_mean               22430.0  3.951048  1.551350  0.500000   
teammate_assists_mean             22430.0  2.511904  1.060460  0.375000   
teammate_goals_per90_mean         22430.0  0.178116  0.062807  0.020337   
teammate_goal_contrib_per90_mean  22430.0  0.291814  0.101148  0.071112   
teammate_count                    22430.0  9.251984  1.580819  4.000000   

                                       25%       50%        75%        max  
teammate_goals_mean               2.857143  3.666667   4.750000  16.333333  
teammate_assists_mean             1.750000  2.300000   3.083333   9.142857  
teammate_goals_per90_mean         0.133491  0.166979   0.210844   0.571426  
teammate_goal_contrib_per90_mean  0.219531  0.272323   0.344083   0.863544  
teammate_count                    8.000000  9.000000  10.000000  15.000000  


In [89]:
train_team[
    [
        "player",
        "team",
        "season",
        "goals",
        "teammate_goals_mean",
        "teammate_goals_per90_mean",
        "teammate_count"
    ]
].head(20)

,player,team,season,goals,teammate_goals_mean,teammate_goals_per90_mean,teammate_count
0,Abdelhafid Tasfaout,Guingamp,2000-2001,5.0,4.833333,0.185133,6
1,Abder Ramdane,Freiburg,2000-2001,3.0,3.800000,0.166577,10
2,Adaílton,Hellas Verona,2000-2001,4.0,2.600000,0.138109,10
3,Ade Akinbiyi,Leicester City,2000-2001,9.0,2.000000,0.095814,9
4,Adel Sellimi,Freiburg,2000-2001,10.0,3.100000,0.136063,10
5,Adrian Mutu,Hellas Verona,2000-2001,4.0,2.600000,0.149480,10
6,Adriano Gabiru,Marseille,2000-2001,3.0,3.428571,0.171333,7
7,Agostinho,Málaga,2000-2001,4.0,5.750000,0.247708,8
8,Aílton Gonçalves,Werder Bremen,2000-2001,14.0,3.200000,0.122082,10
9,Aimo Diana,Brescia,2000-2001,2.0,3.777778,0.154035,9


In [90]:
experiments["Exp8a_teammate_scoring"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "teammate_goals_mean",
            "teammate_goals_per90_mean"
        ]
    ),
    "categorical": context_cols
}

In [91]:
exp = experiments["Exp8a_teammate_scoring"]

result_exp8a = run_experiment(
    experiment_name="Exp8a_teammate_scoring",
    train_df=train_team,
    val_df=val_team,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp8a_teammate_scoring]
Input Dimension : 20
Validation MAE  : 2.4441
Validation RMSE : 3.5143
Validation R²   : 0.4088


In [92]:
experiments["Exp8b_teammate_attack"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "teammate_goals_mean",
            "teammate_assists_mean",
            "teammate_goals_per90_mean",
            "teammate_goal_contrib_per90_mean",
            "teammate_count"
        ]
    ),
    "categorical": context_cols
}

In [93]:
exp = experiments["Exp8b_teammate_attack"]

result_exp8b = run_experiment(
    experiment_name="Exp8b_teammate_attack",
    train_df=train_team,
    val_df=val_team,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp8b_teammate_attack]
Input Dimension : 23
Validation MAE  : 2.4097
Validation RMSE : 3.4909
Validation R²   : 0.4167


In [94]:
print(val_df["next_goals"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
))

count    923.000000
mean       3.198267
std        4.573184
min        0.000000
50%        1.000000
75%        4.000000
90%        9.000000
95%       12.000000
99%       21.000000
max       31.000000
Name: next_goals, dtype: float64


In [97]:
print("10+ goals:", (val_df["next_goals"] >= 10).sum())
print("15+ goals:", (val_df["next_goals"] >= 15).sum())
print("20+ goals:", (val_df["next_goals"] >= 20).sum())
print("25+ goals:", (val_df["next_goals"] >= 25).sum())

10+ goals: 87
15+ goals: 32
20+ goals: 16
25+ goals: 5


In [100]:
exp = experiments["Exp3_basic_per90_context"]

result_exp3_analysis = run_experiment(
    experiment_name="Exp3_analysis",
    train_df=train_df,
    val_df=val_df,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp3_analysis]
Input Dimension : 18
Validation MAE  : 2.4037
Validation RMSE : 3.4829
Validation R²   : 0.4193


In [101]:
val_analysis = val_df[
    [
        "player",
        "team",
        "league",
        "position_group",
        "goals",
        "next_goals"
    ]
].copy()

val_analysis["pred_goals"] = result_exp3_analysis["val_preds"]

val_analysis["error"] = (
    val_analysis["next_goals"]
    - val_analysis["pred_goals"]
)

val_analysis["abs_error"] = (
    val_analysis["error"].abs()
)

In [102]:
val_analysis["goal_band"] = pd.cut(
    val_analysis["next_goals"],
    bins=[-1, 0, 4, 9, 14, 19, 100],
    labels=[
        "0 goals",
        "1-4",
        "5-9",
        "10-14",
        "15-19",
        "20+"
    ]
)

In [103]:
band_result = (
    val_analysis
    .groupby("goal_band", observed=True)
    .agg(
        count=("next_goals", "size"),
        actual_mean=("next_goals", "mean"),
        pred_mean=("pred_goals", "mean"),
        mae=("abs_error", "mean")
    )
)

band_result

,count,actual_mean,pred_mean,mae
goal_band,,,,
0 goals,339,0.000000,1.979451,1.979517
1-4,355,2.140845,2.522536,1.416527
5-9,142,6.591549,4.728287,3.071610
10-14,55,11.363636,6.299380,5.558140
15-19,16,16.312500,8.541168,8.108378
20+,16,23.125000,12.306341,10.818658


In [104]:
val_analysis[
    [
        "player",
        "team",
        "league",
        "position_group",
        "goals",
        "next_goals",
        "pred_goals",
        "error",
        "abs_error"
    ]
].sort_values(
    "abs_error",
    ascending=False
).head(20)

,player,team,league,position_group,goals,next_goals,pred_goals,error,abs_error
580,Mateo Retegui,Genoa,Serie A,FW,7,25.0,5.863905,19.136095,19.136095
703,Ousmane Dembélé,Paris S-G,Ligue 1,FW,3,21.0,3.440472,17.559528,17.559528
716,Patrik Schick,Leverkusen,Bundesliga,FW,7,21.0,5.511709,15.488291,15.488291
106,Ayoze Pérez,Betis,La Liga,FW,9,19.0,4.252932,14.747068,14.747068
631,Mohamed Salah,Liverpool,Premier League,FW,18,29.0,14.344234,14.655766,14.655766
160,Chris Wood,Nott'ham Forest,Premier League,FW,14,20.0,5.886368,14.113632,14.113632
691,Omar Marmoush,Eint Frankfurt,Bundesliga,FW,12,22.0,8.642660,13.357340,13.357340
578,Mason Greenwood,Getafe,La Liga,FW,8,21.0,7.915533,13.084467,13.084467
762,Robert Lewandowski,Barcelona,La Liga,FW,19,27.0,13.927444,13.072556,13.072556
910,Yoane Wissa,Brentford,Premier League,FW,12,19.0,6.338429,12.661571,12.661571


In [105]:
val_analysis[
    val_analysis["next_goals"] >= 15
][
    [
        "player",
        "team",
        "goals",
        "next_goals",
        "pred_goals",
        "abs_error"
    ]
].sort_values(
    "next_goals",
    ascending=False
)

,player,team,goals,next_goals,pred_goals,abs_error
487,Kylian Mbappé,Paris S-G,27,31.0,20.740671,10.259329
631,Mohamed Salah,Liverpool,18,29.0,14.344234,14.655766
762,Robert Lewandowski,Barcelona,19,27.0,13.927444,13.072556
330,Harry Kane,Bayern Munich,36,26.0,25.010035,0.989965
580,Mateo Retegui,Genoa,7,25.0,5.863905,19.136095
38,Alexander Isak,Newcastle Utd,21,23.0,14.455721,8.544279
250,Erling Haaland,Manchester City,27,22.0,20.055702,1.944298
691,Omar Marmoush,Eint Frankfurt,12,22.0,8.642660,13.357340
819,Serhou Guirassy,Stuttgart,28,21.0,18.937605,2.062395
703,Ousmane Dembélé,Paris S-G,3,21.0,3.440472,17.559528


In [106]:
train_matched = train_df[
    train_df["matched_next"] == True
].copy()

val_matched = val_df[
    val_df["matched_next"] == True
].copy()

test_matched = test_df[
    test_df["matched_next"] == True
].copy()

In [107]:
print("Original Train:", train_df.shape)
print("Matched Train :", train_matched.shape)

print()

print("Original Val:", val_df.shape)
print("Matched Val :", val_matched.shape)

print()

print("Original Test:", test_df.shape)
print("Matched Test :", test_matched.shape)

Original Train: (22430, 22)
Matched Train : (18634, 22)

Original Val: (923, 22)
Matched Val : (769, 22)

Original Test: (926, 22)
Matched Test : (751, 22)


In [108]:
print("Original Train target")
print(train_df["next_goals"].describe())

print()

print("Matched Train target")
print(train_matched["next_goals"].describe())

Original Train target
count    22430.000000
mean         2.981275
std          4.509259
min          0.000000
25%          0.000000
50%          1.000000
75%          4.000000
max         50.000000
Name: next_goals, dtype: float64

Matched Train target
count    18634.000000
mean         3.588601
std          4.721892
min          0.000000
25%          0.000000
50%          2.000000
75%          5.000000
max         50.000000
Name: next_goals, dtype: float64


In [109]:
print(
    "Original zero ratio:",
    (train_df["next_goals"] == 0).mean()
)

print(
    "Matched zero ratio:",
    (train_matched["next_goals"] == 0).mean()
)

Original zero ratio: 0.39366919304502895
Matched zero ratio: 0.27015133626703874


In [110]:
exp = experiments["Exp3_basic_per90_context"]

result_matched_exp3 = run_experiment(
    experiment_name="Matched_Exp3",
    train_df=train_matched,
    val_df=val_matched,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Matched_Exp3]
Input Dimension : 18
Validation MAE  : 2.4954
Validation RMSE : 3.5479
Validation R²   : 0.4434


In [111]:
matched_analysis = val_matched[
    [
        "player",
        "team",
        "league",
        "position_group",
        "goals",
        "next_goals"
    ]
].copy()

matched_analysis["pred_goals"] = (
    result_matched_exp3["val_preds"]
)

matched_analysis["error"] = (
    matched_analysis["next_goals"]
    - matched_analysis["pred_goals"]
)

matched_analysis["abs_error"] = (
    matched_analysis["error"].abs()
)

In [112]:
matched_analysis["goal_band"] = pd.cut(
    matched_analysis["next_goals"],
    bins=[-1, 0, 4, 9, 14, 19, 100],
    labels=[
        "0 goals",
        "1-4",
        "5-9",
        "10-14",
        "15-19",
        "20+"
    ]
)

In [113]:
matched_band_result = (
    matched_analysis
    .groupby("goal_band", observed=True)
    .agg(
        count=("next_goals", "size"),
        actual_mean=("next_goals", "mean"),
        pred_mean=("pred_goals", "mean"),
        mae=("abs_error", "mean")
    )
)

matched_band_result

,count,actual_mean,pred_mean,mae
goal_band,,,,
0 goals,185,0.000000,2.057384,2.057384
1-4,355,2.140845,2.852565,1.626206
5-9,142,6.591549,5.404716,2.860298
10-14,55,11.363636,6.925693,5.063853
15-19,16,16.312500,9.390453,7.345617
20+,16,23.125000,13.198869,9.926131


In [114]:
matched_mask = val_df["matched_next"].values

old_preds_matched = (
    result_exp3_analysis["val_preds"][matched_mask]
)

old_targets_matched = (
    result_exp3_analysis["val_targets"][matched_mask]
)

In [115]:
old_matched_mae = mean_absolute_error(
    old_targets_matched,
    old_preds_matched
)

old_matched_rmse = mean_squared_error(
    old_targets_matched,
    old_preds_matched
) ** 0.5

old_matched_r2 = r2_score(
    old_targets_matched,
    old_preds_matched
)

print(f"Full-trained → Matched Val MAE : {old_matched_mae:.4f}")
print(f"Full-trained → Matched Val RMSE: {old_matched_rmse:.4f}")
print(f"Full-trained → Matched Val R²  : {old_matched_r2:.4f}")

print()

print(f"Matched-trained MAE : {result_matched_exp3['mae']:.4f}")
print(f"Matched-trained RMSE: {result_matched_exp3['rmse']:.4f}")
print(f"Matched-trained R²  : {result_matched_exp3['r2']:.4f}")

Full-trained → Matched Val MAE : 2.4461
Full-trained → Matched Val RMSE: 3.5731
Full-trained → Matched Val R²  : 0.4355

Matched-trained MAE : 2.4954
Matched-trained RMSE: 3.5479
Matched-trained R²  : 0.4434


In [116]:
train_combined_matched = train_combined[
    train_combined["matched_next"] == True
].copy()

val_combined_matched = val_combined[
    val_combined["matched_next"] == True
].copy()

test_combined_matched = test_combined[
    test_combined["matched_next"] == True
].copy()

print(train_combined_matched.shape)
print(val_combined_matched.shape)
print(test_combined_matched.shape)

(18634, 34)
(769, 34)
(751, 34)


In [117]:
experiments["Matched_Exp6c_prev_scoring"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "prev_goals",
            "prev_goals_per90",
            "has_prev_season"
        ]
    ),
    "categorical": context_cols
}

In [118]:
exp = experiments["Matched_Exp6c_prev_scoring"]

result_matched_exp6c = run_experiment(
    experiment_name="Matched_Exp6c_prev_scoring",
    train_df=train_combined_matched,
    val_df=val_combined_matched,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Matched_Exp6c_prev_scoring]
Input Dimension : 21
Validation MAE  : 2.4922
Validation RMSE : 3.5696
Validation R²   : 0.4366


In [119]:
experiments["Matched_Exp7a_scoring_trend"] = {
    "numeric": (
        basic_numeric_cols
        + per90_cols
        + [
            "goals_3yr_mean",
            "goals_per90_3yr_mean",
            "goals_3yr_max"
        ]
    ),
    "categorical": context_cols
}

In [120]:
exp = experiments["Matched_Exp7a_scoring_trend"]

result_matched_exp7a = run_experiment(
    experiment_name="Matched_Exp7a_scoring_trend",
    train_df=train_combined_matched,
    val_df=val_combined_matched,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Matched_Exp7a_scoring_trend]
Input Dimension : 21
Validation MAE  : 2.4703
Validation RMSE : 3.5575
Validation R²   : 0.4404


In [121]:
exp9_candidates = pd.DataFrame([
    result_matched_exp3,
    result_matched_exp6c,
    result_matched_exp7a
])

exp9_candidates[
    ["experiment", "input_dim", "mae", "rmse", "r2"]
]

,experiment,input_dim,mae,rmse,r2
0,Matched_Exp3,18,2.495372,3.547946,0.443416
1,Matched_Exp6c_prev_scoring,21,2.492182,3.569588,0.436605
2,Matched_Exp7a_scoring_trend,21,2.470254,3.557461,0.440427


In [122]:
full_all = full_df.copy()

# 나이 파생
full_all = add_age_features(full_all)

# 현재 시즌 공격 파생
full_all = add_current_season_features(full_all)

# PK 파생
full_all = add_penalty_features(full_all)

# 직전 시즌
full_all = add_previous_season_features(full_all)

# 최근 3시즌
full_all = add_rolling_features(full_all)

# 팀/동료 환경
full_all = add_teammate_features(full_all)

In [123]:
fill_cols = [
    "prev_goals",
    "prev_minutes",
    "prev_goals_per90",
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "minutes_3yr_mean",
    "goals_3yr_max"
]

full_all[fill_cols] = full_all[fill_cols].fillna(0)

In [124]:
train_all = full_all[
    (full_all["split"] == "train")
    & (full_all["matched_next"] == True)
].copy()

val_all = full_all[
    (full_all["split"] == "val")
    & (full_all["matched_next"] == True)
].copy()

test_all = full_all[
    (full_all["split"] == "test")
    & (full_all["matched_next"] == True)
].copy()

print(train_all.shape)
print(val_all.shape)
print(test_all.shape)

(18634, 48)
(769, 48)
(751, 48)


In [125]:
all_engineered_numeric = (
    basic_numeric_cols
    + per90_cols
    + [
        # age
        "age_squared",

        # current season
        "goal_contrib",
        "goal_share",

        # penalty
        "penalty_taker",
        "penalty_conversion",
        "non_penalty_goal_ratio",

        # previous season
        "prev_goals",
        "prev_minutes",
        "prev_goals_per90",
        "has_prev_season",

        # rolling history
        "goals_3yr_mean",
        "goals_per90_3yr_mean",
        "minutes_3yr_mean",
        "goals_3yr_max",
        "career_seasons",

        # teammate environment
        "teammate_goals_mean",
        "teammate_assists_mean",
        "teammate_goals_per90_mean",
        "teammate_goal_contrib_per90_mean",
        "teammate_count"
    ]
)

all_engineered_categorical = (
    context_cols
    + ["age_group"]
)

In [126]:
experiments["Exp9A_all_engineered"] = {
    "numeric": all_engineered_numeric,
    "categorical": all_engineered_categorical
}

In [127]:
exp = experiments["Exp9A_all_engineered"]

result_exp9a = run_experiment(
    experiment_name="Exp9A_all_engineered",
    train_df=train_all,
    val_df=val_all,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp9A_all_engineered]
Input Dimension : 43
Validation MAE  : 2.5375
Validation RMSE : 3.6327
Validation R²   : 0.4165


In [128]:
exp = experiments["Exp9A_all_engineered"]

result_exp9a = run_experiment(
    experiment_name="Exp9A_all_engineered",
    train_df=train_all,
    val_df=val_all,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp9A_all_engineered]
Input Dimension : 43
Validation MAE  : 2.5375
Validation RMSE : 3.6327
Validation R²   : 0.4165


In [130]:
selected_engineered_numeric = (
    basic_numeric_cols
    + per90_cols
    + [
        "goals_3yr_mean",
        "goals_per90_3yr_mean",
        "goals_3yr_max"
    ]
)

selected_engineered_categorical = context_cols

In [131]:
experiments["Exp9B_selected_engineered"] = {
    "numeric": selected_engineered_numeric,
    "categorical": selected_engineered_categorical
}

In [132]:
exp = experiments["Exp9B_selected_engineered"]

result_exp9b = run_experiment(
    experiment_name="Exp9B_selected_engineered",
    train_df=train_all,
    val_df=val_all,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp9B_selected_engineered]
Input Dimension : 21
Validation MAE  : 2.4703
Validation RMSE : 3.5575
Validation R²   : 0.4404


In [133]:
exp9_results = pd.DataFrame([
    result_matched_exp3,
    result_exp9a,
    result_exp9b
])

exp9_results[
    ["experiment", "input_dim", "mae", "rmse", "r2"]
]

,experiment,input_dim,mae,rmse,r2
0,Matched_Exp3,18,2.495372,3.547946,0.443416
1,Exp9A_all_engineered,43,2.537493,3.632722,0.416500
2,Exp9B_selected_engineered,21,2.470254,3.557461,0.440427


In [134]:
train_teams = sorted(
    train_all["team"].dropna().unique()
)

team_to_idx = {
    team: idx + 1
    for idx, team in enumerate(train_teams)
}

# 0번은 Train에서 보지 못한 팀용
UNK_TEAM = 0

print("Train team count:", len(team_to_idx))

Train team count: 214


In [135]:
def encode_team(df, team_to_idx):
    return (
        df["team"]
        .map(team_to_idx)
        .fillna(UNK_TEAM)
        .astype(int)
    )

In [136]:
train_team_ids = encode_team(
    train_all,
    team_to_idx
)

val_team_ids = encode_team(
    val_all,
    team_to_idx
)

test_team_ids = encode_team(
    test_all,
    team_to_idx
)

In [137]:
print(
    "Train unknown:",
    (train_team_ids == 0).sum()
)

print(
    "Validation unknown:",
    (val_team_ids == 0).sum()
)

print(
    "Test unknown:",
    (test_team_ids == 0).sum()
)

Train unknown: 0
Validation unknown: 21
Test unknown: 16


In [138]:
print(
    "Validation unknown ratio:",
    (val_team_ids == 0).mean()
)

print(
    "Test unknown ratio:",
    (test_team_ids == 0).mean()
)

Validation unknown ratio: 0.027308192457737322
Test unknown ratio: 0.02130492676431425


In [139]:
numeric_cols_exp10 = selected_engineered_numeric
categorical_cols_exp10 = selected_engineered_categorical

In [140]:
(
    X_train_processed,
    X_val_processed,
    y_train_exp10,
    y_val_exp10,
    preprocessor_exp10
) = prepare_data(
    train_all,
    val_all,
    numeric_cols_exp10,
    categorical_cols_exp10
)

print(X_train_processed.shape)
print(X_val_processed.shape)

(18634, 21)
(769, 21)


In [141]:
X_train_tensor = torch.tensor(
    X_train_processed,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_processed,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_exp10.values,
    dtype=torch.float32
).reshape(-1, 1)

y_val_tensor = torch.tensor(
    y_val_exp10.values,
    dtype=torch.float32
).reshape(-1, 1)

In [142]:
team_train_tensor = torch.tensor(
    train_team_ids.values,
    dtype=torch.long
)

team_val_tensor = torch.tensor(
    val_team_ids.values,
    dtype=torch.long
)

In [143]:
train_dataset_exp10 = TensorDataset(
    X_train_tensor,
    team_train_tensor,
    y_train_tensor
)

val_dataset_exp10 = TensorDataset(
    X_val_tensor,
    team_val_tensor,
    y_val_tensor
)

train_loader_exp10 = DataLoader(
    train_dataset_exp10,
    batch_size=64,
    shuffle=True
)

val_loader_exp10 = DataLoader(
    val_dataset_exp10,
    batch_size=64,
    shuffle=False
)

In [144]:
X_batch, team_batch, y_batch = next(
    iter(train_loader_exp10)
)

print("X:", X_batch.shape)
print("Team:", team_batch.shape)
print("y:", y_batch.shape)

print(team_batch[:10])

X: torch.Size([64, 21])
Team: torch.Size([64])
y: torch.Size([64, 1])
tensor([ 26, 120,  72, 131,  16, 104, 179,  85,  68,  87])


In [145]:
class MLPTeamEmbedding(nn.Module):

    def __init__(
        self,
        input_dim,
        num_teams,
        team_embedding_dim=8
    ):
        super().__init__()

        # 0 = UNK 포함
        self.team_embedding = nn.Embedding(
            num_embeddings=num_teams,
            embedding_dim=team_embedding_dim
        )

        combined_dim = (
            input_dim + team_embedding_dim
        )

        self.fc1 = nn.Linear(combined_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.relu = nn.ReLU()


    def forward(self, x, team_id):

        # [batch]
        # ↓
        # [batch, 8]
        team_emb = self.team_embedding(team_id)

        # 일반 피처와 embedding 연결
        x = torch.cat(
            [x, team_emb],
            dim=1
        )

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [146]:
input_dim = X_train_processed.shape[1]
num_teams = len(team_to_idx) + 1

model_exp10 = MLPTeamEmbedding(
    input_dim=input_dim,
    num_teams=num_teams,
    team_embedding_dim=8
).to(device)

print(model_exp10)

MLPTeamEmbedding(
  (team_embedding): Embedding(215, 8)
  (fc1): Linear(in_features=29, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)


In [147]:
X_batch, team_batch, y_batch = next(
    iter(train_loader_exp10)
)

X_batch = X_batch.to(device)
team_batch = team_batch.to(device)

pred = model_exp10(
    X_batch,
    team_batch
)

print("X shape:", X_batch.shape)
print("Team ID shape:", team_batch.shape)
print("Prediction shape:", pred.shape)

X shape: torch.Size([64, 21])
Team ID shape: torch.Size([64])
Prediction shape: torch.Size([64, 1])


In [148]:
def train_team_embedding_model(
    model,
    train_loader,
    val_loader,
    epochs=50,
    lr=0.001
):
    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # =====================
        # Train
        # =====================
        model.train()

        train_loss = 0.0

        for X_batch, team_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            team_batch = team_batch.to(device)
            y_batch = y_batch.to(device)

            pred = model(
                X_batch,
                team_batch
            )

            loss = criterion(pred, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)


        # =====================
        # Validation
        # =====================
        model.eval()

        val_loss = 0.0

        with torch.no_grad():
            for X_batch, team_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                team_batch = team_batch.to(device)
                y_batch = y_batch.to(device)

                pred = model(
                    X_batch,
                    team_batch
                )

                loss = criterion(pred, y_batch)

                val_loss += loss.item()

        val_loss /= len(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

    return model, train_losses, val_losses

In [149]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [152]:
model_exp10 = MLPTeamEmbedding(
    input_dim=X_train_processed.shape[1],
    num_teams=len(team_to_idx) + 1,
    team_embedding_dim=8
).to(device)

In [153]:
model_exp10, train_losses_exp10, val_losses_exp10 = (
    train_team_embedding_model(
        model=model_exp10,
        train_loader=train_loader_exp10,
        val_loader=val_loader_exp10,
        epochs=50,
        lr=0.001
    )
)

In [154]:
model_exp10.eval()

val_preds_exp10 = []
val_targets_exp10 = []

with torch.no_grad():
    for X_batch, team_batch, y_batch in val_loader_exp10:
        X_batch = X_batch.to(device)
        team_batch = team_batch.to(device)

        pred = model_exp10(
            X_batch,
            team_batch
        )

        val_preds_exp10.extend(
            pred.cpu().numpy().flatten()
        )

        val_targets_exp10.extend(
            y_batch.numpy().flatten()
        )

In [155]:
exp10_mae = mean_absolute_error(
    val_targets_exp10,
    val_preds_exp10
)

exp10_rmse = mean_squared_error(
    val_targets_exp10,
    val_preds_exp10
) ** 0.5

exp10_r2 = r2_score(
    val_targets_exp10,
    val_preds_exp10
)

print("[Exp10 Team Embedding]")
print(f"Validation MAE  : {exp10_mae:.4f}")
print(f"Validation RMSE : {exp10_rmse:.4f}")
print(f"Validation R²   : {exp10_r2:.4f}")

[Exp10 Team Embedding]
Validation MAE  : 2.6026
Validation RMSE : 3.7388
Validation R²   : 0.3819


In [156]:
team_stats_df = pd.read_csv(
    "../data/team_season_stats_2000_2024.csv"
)

team_alias_df = pd.read_csv(
    "../data/team_name_alias_map.csv"
)

print(team_stats_df.shape)
display(team_stats_df.head())

print()
print(team_alias_df.shape)
display(team_alias_df.head())

(2434, 22)


,league,season,team_name,position,played,wins,draws,losses,goals_for,goals_against,...,team_goals_per_game,team_goals_against_per_game,team_points_per_game,team_goal_diff_per_game,team_win_rate,league_team_count,team_rank_pct,league_avg_goals_per_game,team_attack_strength,team_defense_strength
0,Bundesliga,2000-2001,Bayern Munich,1,34,19,6,9,62,37,...,1.823529,1.088235,1.852941,0.735294,0.558824,18,1.000000,1.465686,1.244147,1.346847
1,Bundesliga,2000-2001,Schalke 04,2,34,18,8,8,65,35,...,1.911765,1.029412,1.823529,0.882353,0.529412,18,0.941176,1.465686,1.304348,1.423810
2,Bundesliga,2000-2001,Dortmund,3,34,16,10,8,62,42,...,1.823529,1.235294,1.705882,0.588235,0.470588,18,0.882353,1.465686,1.244147,1.186508
3,Bundesliga,2000-2001,Leverkusen,4,34,17,6,11,54,40,...,1.588235,1.176471,1.676471,0.411765,0.500000,18,0.823529,1.465686,1.083612,1.245833
4,Bundesliga,2000-2001,Hertha,5,34,18,2,14,58,52,...,1.705882,1.529412,1.647059,0.176471,0.529412,18,0.764706,1.465686,1.163880,0.958333



(66, 2)


,player_data_team,standings_team
0,1860 Munich,Munich 1860
1,AA Aachen,Aachen
2,Alavés,Alaves
3,Almería,Almeria
4,Arles-Avignon,Arles


In [157]:
team_name_map = dict(
    zip(
        team_alias_df["player_data_team"],
        team_alias_df["standings_team"]
    )
)

In [158]:
def normalize_team_name(df):
    df = df.copy()

    df["team_name"] = (
        df["team"]
        .map(team_name_map)
        .fillna(df["team"])
    )

    return df

In [161]:
train_team = normalize_team_name(train_all)
val_team = normalize_team_name(val_all)
test_team = normalize_team_name(test_all)

In [162]:
train_team[
    ["team", "team_name"]
].drop_duplicates().head(20)

,team,team_name
18684,Brighton,Brighton
13697,West Ham,West Ham
5541,Werder Bremen,Werder Bremen
14790,Hamburger SV,Hamburg
4588,Tottenham,Tottenham
13698,Everton,Everton
16775,Burnley,Burnley
17729,Toulouse,Toulouse
3630,Blackburn,Blackburn
16776,Huddersfield,Huddersfield


In [163]:
team_feature_cols = [
    "position",
    "goals_for",
    "goals_against",
    "goal_difference",
    "points",
    "team_goals_per_game",
    "team_goals_against_per_game",
    "team_points_per_game",
    "team_goal_diff_per_game",
    "team_win_rate",
    "team_rank_pct",
    "team_attack_strength",
    "team_defense_strength"
]

In [164]:
merge_cols = [
    "league",
    "season",
    "team_name"
]

team_stats_for_merge = team_stats_df[
    merge_cols + team_feature_cols
].copy()

In [165]:
train_team = train_team.merge(
    team_stats_for_merge,
    on=merge_cols,
    how="left"
)

val_team = val_team.merge(
    team_stats_for_merge,
    on=merge_cols,
    how="left"
)

test_team = test_team.merge(
    team_stats_for_merge,
    on=merge_cols,
    how="left"
)

In [168]:
print(train_team.columns.tolist())

['player', 'nation', 'team', 'league', 'season', 'target_season', 'position_x', 'position_group', 'age', 'matched_next', 'starts', 'minutes', 'goals', 'assists', 'non_penalty_goals', 'penalty_goals', 'penalty_attempts', 'goals_per90', 'assists_per90', 'goal_contrib_per90', 'next_goals', 'next_10plus', 'split', 'age_squared', 'age_group', 'minutes_per_start', 'goal_contrib', 'goal_share', 'assist_share', 'penalty_conversion', 'non_penalty_goal_ratio', 'penalty_taker', 'season_start', 'prev_season_start', 'prev_goals', 'prev_minutes', 'prev_goals_per90', 'has_prev_season', 'goals_3yr_mean', 'goals_per90_3yr_mean', 'minutes_3yr_mean', 'goals_3yr_max', 'career_seasons', 'teammate_goals_mean', 'teammate_assists_mean', 'teammate_goals_per90_mean', 'teammate_goal_contrib_per90_mean', 'teammate_count', 'team_name', 'position_y', 'goals_for', 'goals_against', 'goal_difference', 'points', 'team_goals_per_game', 'team_goals_against_per_game', 'team_points_per_game', 'team_goal_diff_per_game', 'te

In [169]:
def fix_position_columns(df):
    df = df.copy()

    df = df.rename(
        columns={
            "position_x": "position",
            "position_y": "team_rank"
        }
    )

    return df


train_team = fix_position_columns(train_team)
val_team = fix_position_columns(val_team)
test_team = fix_position_columns(test_team)

In [170]:
print(
    [col for col in train_team.columns
     if "position" in col or "rank" in col]
)

['position', 'position_group', 'team_rank', 'team_rank_pct']


In [171]:
print(
    "Train match:",
    train_team["team_rank"].notna().mean()
)

print(
    "Validation match:",
    val_team["team_rank"].notna().mean()
)

print(
    "Test match:",
    test_team["team_rank"].notna().mean()
)

Train match: 1.0
Validation match: 0.9830949284785435
Test match: 0.9920106524633822


In [172]:
for name, df in [
    ("Train", train_team),
    ("Validation", val_team),
    ("Test", test_team)
]:
    unmatched = (
        df[df["team_rank"].isna()]
        [["season", "league", "team", "team_name"]]
        .drop_duplicates()
    )

    print(f"{name} unmatched teams: {len(unmatched)}")
    display(unmatched)

Train unmatched teams: 0


,season,league,team,team_name


Validation unmatched teams: 2


,season,league,team,team_name
16,2023-2024,Bundesliga,Gladbach,Gladbach
18,2023-2024,Premier League,Luton Town,Luton Town


Test unmatched teams: 1


,season,league,team,team_name
240,2024-2025,Bundesliga,Gladbach,Gladbach


In [173]:
extra_team_aliases = {
    "Gladbach": "M'gladbach",
    "Luton Town": "Luton"
}

team_name_map.update(extra_team_aliases)

In [174]:
train_team = normalize_team_name(train_all)
val_team = normalize_team_name(val_all)
test_team = normalize_team_name(test_all)

In [175]:
team_stats_for_merge = team_stats_df[
    [
        "league",
        "season",
        "team_name",
        "position",
        "goals_for",
        "goals_against",
        "goal_difference",
        "points",
        "team_goals_per_game",
        "team_goals_against_per_game",
        "team_points_per_game",
        "team_goal_diff_per_game",
        "team_win_rate",
        "team_rank_pct",
        "team_attack_strength",
        "team_defense_strength"
    ]
].copy()

team_stats_for_merge = team_stats_for_merge.rename(
    columns={"position": "team_rank"}
)

In [176]:
merge_cols = ["league", "season", "team_name"]

train_team = train_team.merge(
    team_stats_for_merge,
    on=merge_cols,
    how="left"
)

val_team = val_team.merge(
    team_stats_for_merge,
    on=merge_cols,
    how="left"
)

test_team = test_team.merge(
    team_stats_for_merge,
    on=merge_cols,
    how="left"
)

In [177]:
print(
    "Train match:",
    train_team["team_rank"].notna().mean()
)

print(
    "Validation match:",
    val_team["team_rank"].notna().mean()
)

print(
    "Test match:",
    test_team["team_rank"].notna().mean()
)

Train match: 1.0
Validation match: 1.0
Test match: 1.0


In [178]:
team_performance_cols = [
    "team_rank_pct",
    "team_goals_per_game",
    "team_points_per_game",
    "team_goal_diff_per_game",
    "team_attack_strength"
]

In [179]:
exp9c_numeric = (
    selected_engineered_numeric
    + team_performance_cols
)

exp9c_categorical = selected_engineered_categorical

In [180]:
experiments["Exp9C_team_performance"] = {
    "numeric": exp9c_numeric,
    "categorical": exp9c_categorical
}

In [181]:
exp = experiments["Exp9C_team_performance"]

result_exp9c = run_experiment(
    experiment_name="Exp9C_team_performance",
    train_df=train_team,
    val_df=val_team,
    numeric_cols=exp["numeric"],
    categorical_cols=exp["categorical"]
)


[Exp9C_team_performance]
Input Dimension : 26
Validation MAE  : 2.4592
Validation RMSE : 3.5450
Validation R²   : 0.4444


In [182]:
exp9_team_compare = pd.DataFrame([
    result_matched_exp3,
    result_exp9b,
    result_exp9c
])

exp9_team_compare[
    ["experiment", "input_dim", "mae", "rmse", "r2"]
]

,experiment,input_dim,mae,rmse,r2
0,Matched_Exp3,18,2.495372,3.547946,0.443416
1,Exp9B_selected_engineered,21,2.470254,3.557461,0.440427
2,Exp9C_team_performance,26,2.459196,3.544961,0.444352


In [183]:
team_attack_cols = [
    "team_goals_per_game",
    "team_attack_strength"
]

exp9ca_numeric = (
    selected_engineered_numeric
    + team_attack_cols
)

result_exp9ca = run_experiment(
    experiment_name="Exp9Ca_team_attack",
    train_df=train_team,
    val_df=val_team,
    numeric_cols=exp9ca_numeric,
    categorical_cols=selected_engineered_categorical
)


[Exp9Ca_team_attack]
Input Dimension : 23
Validation MAE  : 2.4802
Validation RMSE : 3.5374
Validation R²   : 0.4467


In [184]:
team_strength_cols = [
    "team_rank_pct",
    "team_points_per_game",
    "team_goal_diff_per_game"
]

exp9cb_numeric = (
    selected_engineered_numeric
    + team_strength_cols
)

result_exp9cb = run_experiment(
    experiment_name="Exp9Cb_team_strength",
    train_df=train_team,
    val_df=val_team,
    numeric_cols=exp9cb_numeric,
    categorical_cols=selected_engineered_categorical
)


[Exp9Cb_team_strength]
Input Dimension : 24
Validation MAE  : 2.4412
Validation RMSE : 3.5323
Validation R²   : 0.4483


In [185]:
team_feature_compare = pd.DataFrame([
    result_exp9b,
    result_exp9ca,
    result_exp9cb,
    result_exp9c
])

team_feature_compare[
    ["experiment", "input_dim", "mae", "rmse", "r2"]
]

,experiment,input_dim,mae,rmse,r2
0,Exp9B_selected_engineered,21,2.470254,3.557461,0.440427
1,Exp9Ca_team_attack,23,2.480224,3.537374,0.446728
2,Exp9Cb_team_strength,24,2.441230,3.532324,0.448307
3,Exp9C_team_performance,26,2.459196,3.544961,0.444352


In [186]:
final_feature_numeric = (
    selected_engineered_numeric
    + [
        "team_rank_pct",
        "team_points_per_game",
        "team_goal_diff_per_game"
    ]
)

final_feature_categorical = (
    selected_engineered_categorical
)

In [187]:
final_feature_numeric = (
    selected_engineered_numeric
    + [
        "team_rank_pct",
        "team_points_per_game",
        "team_goal_diff_per_game"
    ]
)

final_feature_categorical = (
    selected_engineered_categorical
)

print("Numeric:", len(final_feature_numeric))
print("Categorical:", final_feature_categorical)

Numeric: 17
Categorical: ['league', 'position_group']


In [189]:
(
    X_train_final,
    X_val_final,
    y_train_final,
    y_val_final,
    preprocessor_final
) = prepare_data(
    train_team,
    val_team,
    final_feature_numeric,
    final_feature_categorical
)

print("X Train:", X_train_final.shape)
print("X Val  :", X_val_final.shape)

X Train: (18634, 24)
X Val  : (769, 24)


In [190]:
train_teams = sorted(
    train_team["team_name"].dropna().unique()
)

team_to_idx = {
    team: idx + 1
    for idx, team in enumerate(train_teams)
}

UNK_TEAM = 0


def encode_team_final(df):
    return (
        df["team_name"]
        .map(team_to_idx)
        .fillna(UNK_TEAM)
        .astype(int)
    )


train_team_ids = encode_team_final(train_team)
val_team_ids = encode_team_final(val_team)
test_team_ids = encode_team_final(test_team)

print("Train teams:", len(team_to_idx))
print("Val Team UNK:", (val_team_ids == 0).sum())
print("Val Team UNK ratio:", (val_team_ids == 0).mean())

Train teams: 214
Val Team UNK: 11
Val Team UNK ratio: 0.014304291287386216


In [191]:
train_players = sorted(
    train_team["player"].dropna().unique()
)

player_to_idx = {
    player: idx + 1
    for idx, player in enumerate(train_players)
}

UNK_PLAYER = 0

print("Train player count:", len(player_to_idx))

Train player count: 4905


In [192]:
def encode_player(df):
    return (
        df["player"]
        .map(player_to_idx)
        .fillna(UNK_PLAYER)
        .astype(int)
    )


train_player_ids = encode_player(train_team)
val_player_ids = encode_player(val_team)
test_player_ids = encode_player(test_team)

In [193]:
print(
    "Train Player unknown:",
    (train_player_ids == 0).sum()
)

print(
    "Validation Player unknown:",
    (val_player_ids == 0).sum()
)

print(
    "Test Player unknown:",
    (test_player_ids == 0).sum()
)

print()

print(
    "Validation Player unknown ratio:",
    (val_player_ids == 0).mean()
)

print(
    "Test Player unknown ratio:",
    (test_player_ids == 0).mean()
)

Train Player unknown: 0
Validation Player unknown: 207
Test Player unknown: 304

Validation Player unknown ratio: 0.26918075422626786
Test Player unknown ratio: 0.4047936085219707


In [195]:
X_train_tensor = torch.tensor(
    X_train_final,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_final,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    np.asarray(y_train_final),
    dtype=torch.float32
).reshape(-1, 1)

y_val_tensor = torch.tensor(
    np.asarray(y_val_final),
    dtype=torch.float32
).reshape(-1, 1)


team_train_tensor = torch.tensor(
    train_team_ids.values,
    dtype=torch.long
)

team_val_tensor = torch.tensor(
    val_team_ids.values,
    dtype=torch.long
)


player_train_tensor = torch.tensor(
    train_player_ids.values,
    dtype=torch.long
)

player_val_tensor = torch.tensor(
    val_player_ids.values,
    dtype=torch.long
)

In [196]:
print("X:", X_train_tensor.shape)
print("Team:", team_train_tensor.shape)
print("Player:", player_train_tensor.shape)
print("y:", y_train_tensor.shape)

X: torch.Size([18634, 24])
Team: torch.Size([18634])
Player: torch.Size([18634])
y: torch.Size([18634, 1])


In [197]:
class MLPTeamEmbeddingFinal(nn.Module):

    def __init__(
        self,
        input_dim,
        num_teams,
        team_embedding_dim=8
    ):
        super().__init__()

        self.team_embedding = nn.Embedding(
            num_embeddings=num_teams,
            embedding_dim=team_embedding_dim,
            padding_idx=0
        )

        combined_dim = input_dim + team_embedding_dim

        self.fc1 = nn.Linear(combined_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.relu = nn.ReLU()

    def forward(self, x, team_id):

        team_emb = self.team_embedding(team_id)

        x = torch.cat(
            [x, team_emb],
            dim=1
        )

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [198]:
train_dataset_exp10 = TensorDataset(
    X_train_tensor,
    team_train_tensor,
    y_train_tensor
)

val_dataset_exp10 = TensorDataset(
    X_val_tensor,
    team_val_tensor,
    y_val_tensor
)

train_loader_exp10 = DataLoader(
    train_dataset_exp10,
    batch_size=64,
    shuffle=True
)

val_loader_exp10 = DataLoader(
    val_dataset_exp10,
    batch_size=64,
    shuffle=False
)

In [200]:
class MLPPlayerEmbedding(nn.Module):

    def __init__(
        self,
        input_dim,
        num_players,
        player_embedding_dim=16
    ):
        super().__init__()

        self.player_embedding = nn.Embedding(
            num_embeddings=num_players,
            embedding_dim=player_embedding_dim,
            padding_idx=0
        )

        combined_dim = (
            input_dim
            + player_embedding_dim
        )

        self.fc1 = nn.Linear(combined_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.relu = nn.ReLU()

    def forward(self, x, player_id):

        player_emb = self.player_embedding(
            player_id
        )

        x = torch.cat(
            [x, player_emb],
            dim=1
        )

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [201]:
train_dataset_exp11 = TensorDataset(
    X_train_tensor,
    player_train_tensor,
    y_train_tensor
)

val_dataset_exp11 = TensorDataset(
    X_val_tensor,
    player_val_tensor,
    y_val_tensor
)

train_loader_exp11 = DataLoader(
    train_dataset_exp11,
    batch_size=64,
    shuffle=True
)

val_loader_exp11 = DataLoader(
    val_dataset_exp11,
    batch_size=64,
    shuffle=False
)

In [202]:
class MLPTeamPlayerEmbedding(nn.Module):

    def __init__(
        self,
        input_dim,
        num_teams,
        num_players,
        team_embedding_dim=8,
        player_embedding_dim=16
    ):
        super().__init__()

        self.team_embedding = nn.Embedding(
            num_embeddings=num_teams,
            embedding_dim=team_embedding_dim,
            padding_idx=0
        )

        self.player_embedding = nn.Embedding(
            num_embeddings=num_players,
            embedding_dim=player_embedding_dim,
            padding_idx=0
        )

        combined_dim = (
            input_dim
            + team_embedding_dim
            + player_embedding_dim
        )

        self.fc1 = nn.Linear(combined_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.relu = nn.ReLU()

    def forward(
        self,
        x,
        team_id,
        player_id
    ):

        team_emb = self.team_embedding(
            team_id
        )

        player_emb = self.player_embedding(
            player_id
        )

        x = torch.cat(
            [
                x,
                team_emb,
                player_emb
            ],
            dim=1
        )

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [203]:
train_dataset_exp12 = TensorDataset(
    X_train_tensor,
    team_train_tensor,
    player_train_tensor,
    y_train_tensor
)

val_dataset_exp12 = TensorDataset(
    X_val_tensor,
    team_val_tensor,
    player_val_tensor,
    y_val_tensor
)

train_loader_exp12 = DataLoader(
    train_dataset_exp12,
    batch_size=64,
    shuffle=True
)

val_loader_exp12 = DataLoader(
    val_dataset_exp12,
    batch_size=64,
    shuffle=False
)

In [204]:
train_dataset_exp12 = TensorDataset(
    X_train_tensor,
    team_train_tensor,
    player_train_tensor,
    y_train_tensor
)

val_dataset_exp12 = TensorDataset(
    X_val_tensor,
    team_val_tensor,
    player_val_tensor,
    y_val_tensor
)

train_loader_exp12 = DataLoader(
    train_dataset_exp12,
    batch_size=64,
    shuffle=True
)

val_loader_exp12 = DataLoader(
    val_dataset_exp12,
    batch_size=64,
    shuffle=False
)

In [205]:
def reset_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [206]:
def train_embedding_model(
    model,
    train_loader,
    val_loader,
    mode,
    epochs=50,
    lr=0.001
):
    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # =========================
        # Train
        # =========================
        model.train()

        train_loss = 0.0

        for batch in train_loader:

            optimizer.zero_grad()

            if mode == "team":

                X_batch, team_batch, y_batch = batch

                X_batch = X_batch.to(device)
                team_batch = team_batch.to(device)
                y_batch = y_batch.to(device)

                pred = model(
                    X_batch,
                    team_batch
                )

            elif mode == "player":

                X_batch, player_batch, y_batch = batch

                X_batch = X_batch.to(device)
                player_batch = player_batch.to(device)
                y_batch = y_batch.to(device)

                pred = model(
                    X_batch,
                    player_batch
                )

            elif mode == "team_player":

                X_batch, team_batch, player_batch, y_batch = batch

                X_batch = X_batch.to(device)
                team_batch = team_batch.to(device)
                player_batch = player_batch.to(device)
                y_batch = y_batch.to(device)

                pred = model(
                    X_batch,
                    team_batch,
                    player_batch
                )

            else:
                raise ValueError(
                    "mode must be team, player, or team_player"
                )

            loss = criterion(
                pred,
                y_batch
            )

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)


        # =========================
        # Validation
        # =========================
        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for batch in val_loader:

                if mode == "team":

                    X_batch, team_batch, y_batch = batch

                    X_batch = X_batch.to(device)
                    team_batch = team_batch.to(device)
                    y_batch = y_batch.to(device)

                    pred = model(
                        X_batch,
                        team_batch
                    )

                elif mode == "player":

                    X_batch, player_batch, y_batch = batch

                    X_batch = X_batch.to(device)
                    player_batch = player_batch.to(device)
                    y_batch = y_batch.to(device)

                    pred = model(
                        X_batch,
                        player_batch
                    )

                elif mode == "team_player":

                    X_batch, team_batch, player_batch, y_batch = batch

                    X_batch = X_batch.to(device)
                    team_batch = team_batch.to(device)
                    player_batch = player_batch.to(device)
                    y_batch = y_batch.to(device)

                    pred = model(
                        X_batch,
                        team_batch,
                        player_batch
                    )

                loss = criterion(
                    pred,
                    y_batch
                )

                val_loss += loss.item()

        val_loss /= len(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)


    return (
        model,
        train_losses,
        val_losses
    )

In [207]:
def evaluate_embedding_model(
    model,
    val_loader,
    mode
):
    model.eval()

    predictions = []
    targets = []

    with torch.no_grad():

        for batch in val_loader:

            if mode == "team":

                X_batch, team_batch, y_batch = batch

                pred = model(
                    X_batch.to(device),
                    team_batch.to(device)
                )

            elif mode == "player":

                X_batch, player_batch, y_batch = batch

                pred = model(
                    X_batch.to(device),
                    player_batch.to(device)
                )

            elif mode == "team_player":

                X_batch, team_batch, player_batch, y_batch = batch

                pred = model(
                    X_batch.to(device),
                    team_batch.to(device),
                    player_batch.to(device)
                )

            predictions.extend(
                pred.cpu().numpy().flatten()
            )

            targets.extend(
                y_batch.numpy().flatten()
            )


    predictions = np.array(predictions)
    targets = np.array(targets)

    mae = mean_absolute_error(
        targets,
        predictions
    )

    rmse = mean_squared_error(
        targets,
        predictions
    ) ** 0.5

    r2 = r2_score(
        targets,
        predictions
    )

    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "val_preds": predictions,
        "val_targets": targets
    }

In [208]:
reset_seed(SEED)

model_exp10_final = MLPTeamEmbeddingFinal(
    input_dim=X_train_final.shape[1],
    num_teams=len(team_to_idx) + 1,
    team_embedding_dim=8
).to(device)

In [209]:
(
    model_exp10_final,
    train_loss_exp10,
    val_loss_exp10
) = train_embedding_model(
    model=model_exp10_final,
    train_loader=train_loader_exp10,
    val_loader=val_loader_exp10,
    mode="team",
    epochs=50,
    lr=0.001
)

In [210]:
result_exp10_final = evaluate_embedding_model(
    model_exp10_final,
    val_loader_exp10,
    mode="team"
)

print("[Exp10 Final + Team Embedding]")
print(f"Validation MAE  : {result_exp10_final['mae']:.4f}")
print(f"Validation RMSE : {result_exp10_final['rmse']:.4f}")
print(f"Validation R²   : {result_exp10_final['r2']:.4f}")

[Exp10 Final + Team Embedding]
Validation MAE  : 2.6535
Validation RMSE : 3.7073
Validation R²   : 0.3923


In [213]:
reset_seed(SEED)

model_exp11 = MLPPlayerEmbedding(
    input_dim=X_train_final.shape[1],
    num_players=len(player_to_idx) + 1,
    player_embedding_dim=16
).to(device)

In [214]:
(
    model_exp11,
    train_loss_exp11,
    val_loss_exp11
) = train_embedding_model(
    model=model_exp11,
    train_loader=train_loader_exp11,
    val_loader=val_loader_exp11,
    mode="player",
    epochs=50,
    lr=0.001
)

In [215]:
result_exp11 = evaluate_embedding_model(
    model_exp11,
    val_loader_exp11,
    mode="player"
)

print("[Exp11 Final + Player Embedding]")
print(f"Validation MAE  : {result_exp11['mae']:.4f}")
print(f"Validation RMSE : {result_exp11['rmse']:.4f}")
print(f"Validation R²   : {result_exp11['r2']:.4f}")

[Exp11 Final + Player Embedding]
Validation MAE  : 3.2786
Validation RMSE : 4.6906
Validation R²   : 0.0272


In [216]:
reset_seed(SEED)

model_exp12 = MLPTeamPlayerEmbedding(
    input_dim=X_train_final.shape[1],
    num_teams=len(team_to_idx) + 1,
    num_players=len(player_to_idx) + 1,
    team_embedding_dim=8,
    player_embedding_dim=16
).to(device)

In [217]:
(
    model_exp12,
    train_loss_exp12,
    val_loss_exp12
) = train_embedding_model(
    model=model_exp12,
    train_loader=train_loader_exp12,
    val_loader=val_loader_exp12,
    mode="team_player",
    epochs=50,
    lr=0.001
)

In [218]:
result_exp12 = evaluate_embedding_model(
    model_exp12,
    val_loader_exp12,
    mode="team_player"
)

print("[Exp12 Final + Team + Player Embedding]")
print(f"Validation MAE  : {result_exp12['mae']:.4f}")
print(f"Validation RMSE : {result_exp12['rmse']:.4f}")
print(f"Validation R²   : {result_exp12['r2']:.4f}")

[Exp12 Final + Team + Player Embedding]
Validation MAE  : 3.4122
Validation RMSE : 4.7668
Validation R²   : -0.0047


In [219]:
embedding_compare = pd.DataFrame([
    {
        "experiment": "Exp9Cb_tabular",
        "representation_dim": 24,
        "mae": result_exp9cb["mae"],
        "rmse": result_exp9cb["rmse"],
        "r2": result_exp9cb["r2"]
    },
    {
        "experiment": "Exp10_team_embedding",
        "representation_dim": 24 + 8,
        "mae": result_exp10_final["mae"],
        "rmse": result_exp10_final["rmse"],
        "r2": result_exp10_final["r2"]
    },
    {
        "experiment": "Exp11_player_embedding",
        "representation_dim": 24 + 16,
        "mae": result_exp11["mae"],
        "rmse": result_exp11["rmse"],
        "r2": result_exp11["r2"]
    },
    {
        "experiment": "Exp12_team_player_embedding",
        "representation_dim": 24 + 8 + 16,
        "mae": result_exp12["mae"],
        "rmse": result_exp12["rmse"],
        "r2": result_exp12["r2"]
    }
])

embedding_compare

,experiment,representation_dim,mae,rmse,r2
0,Exp9Cb_tabular,24,2.441230,3.532324,0.448307
1,Exp10_team_embedding,32,2.653478,3.707303,0.392295
2,Exp11_player_embedding,40,3.278640,4.690555,0.027196
3,Exp12_team_player_embedding,48,3.412192,4.766845,-0.004706


In [220]:
def summarize_losses(name, train_losses, val_losses):
    best_epoch = np.argmin(val_losses) + 1

    print(f"[{name}]")
    print(f"Final Train Loss : {train_losses[-1]:.4f}")
    print(f"Final Val Loss   : {val_losses[-1]:.4f}")
    print(f"Best Val Loss    : {min(val_losses):.4f}")
    print(f"Best Epoch       : {best_epoch}")
    print()


summarize_losses(
    "Exp10 Team",
    train_loss_exp10,
    val_loss_exp10
)

summarize_losses(
    "Exp11 Player",
    train_loss_exp11,
    val_loss_exp11
)

summarize_losses(
    "Exp12 Team+Player",
    train_loss_exp12,
    val_loss_exp12
)

[Exp10 Team]
Final Train Loss : 9.2579
Final Val Loss   : 14.5413
Best Val Loss    : 12.4701
Best Epoch       : 10

[Exp11 Player]
Final Train Loss : 3.4905
Final Val Loss   : 21.2158
Best Val Loss    : 12.4066
Best Epoch       : 1

[Exp12 Team+Player]
Final Train Loss : 2.9485
Final Val Loss   : 27.2217
Best Val Loss    : 12.5907
Best Epoch       : 1



In [221]:
val_player_ids_np = val_player_ids.to_numpy()

seen_mask = val_player_ids_np != 0
unseen_mask = val_player_ids_np == 0

exp11_preds = result_exp11["val_preds"]
exp11_targets = result_exp11["val_targets"]

In [222]:
def evaluate_group(name, y_true, y_pred, mask):

    mae = mean_absolute_error(
        y_true[mask],
        y_pred[mask]
    )

    rmse = mean_squared_error(
        y_true[mask],
        y_pred[mask]
    ) ** 0.5

    r2 = r2_score(
        y_true[mask],
        y_pred[mask]
    )

    print(
        f"{name:<10} "
        f"N={mask.sum():>3} | "
        f"MAE={mae:.4f} | "
        f"RMSE={rmse:.4f} | "
        f"R²={r2:.4f}"
    )

In [223]:
evaluate_group(
    "Seen",
    exp11_targets,
    exp11_preds,
    seen_mask
)

evaluate_group(
    "Unseen",
    exp11_targets,
    exp11_preds,
    unseen_mask
)

Seen       N=562 | MAE=3.2785 | RMSE=4.7352 | R²=0.1199
Unseen     N=207 | MAE=3.2790 | RMSE=4.5671 | R²=-0.4848


In [224]:
exp9_preds = result_exp9cb["val_preds"]
exp9_targets = result_exp9cb["val_targets"]

print("[Exp9Cb]")

evaluate_group(
    "Seen",
    exp9_targets,
    exp9_preds,
    seen_mask
)

evaluate_group(
    "Unseen",
    exp9_targets,
    exp9_preds,
    unseen_mask
)

[Exp9Cb]
Seen       N=562 | MAE=2.5435 | RMSE=3.7017 | R²=0.4622
Unseen     N=207 | MAE=2.1636 | RMSE=3.0251 | R²=0.3486


In [225]:
import copy

def train_embedding_model_best(
    model,
    train_loader,
    val_loader,
    mode,
    epochs=50,
    lr=0.001
):
    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    train_losses = []
    val_losses = []

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None

    for epoch in range(epochs):

        # ======================
        # Train
        # ======================
        model.train()
        train_loss = 0.0

        for batch in train_loader:

            optimizer.zero_grad()

            if mode == "team":
                X_batch, team_batch, y_batch = batch

                pred = model(
                    X_batch.to(device),
                    team_batch.to(device)
                )

            elif mode == "player":
                X_batch, player_batch, y_batch = batch

                pred = model(
                    X_batch.to(device),
                    player_batch.to(device)
                )

            elif mode == "team_player":
                X_batch, team_batch, player_batch, y_batch = batch

                pred = model(
                    X_batch.to(device),
                    team_batch.to(device),
                    player_batch.to(device)
                )

            y_batch = y_batch.to(device)

            loss = criterion(pred, y_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)


        # ======================
        # Validation
        # ======================
        model.eval()
        val_loss = 0.0

        with torch.no_grad():

            for batch in val_loader:

                if mode == "team":
                    X_batch, team_batch, y_batch = batch

                    pred = model(
                        X_batch.to(device),
                        team_batch.to(device)
                    )

                elif mode == "player":
                    X_batch, player_batch, y_batch = batch

                    pred = model(
                        X_batch.to(device),
                        player_batch.to(device)
                    )

                elif mode == "team_player":
                    X_batch, team_batch, player_batch, y_batch = batch

                    pred = model(
                        X_batch.to(device),
                        team_batch.to(device),
                        player_batch.to(device)
                    )

                y_batch = y_batch.to(device)

                loss = criterion(pred, y_batch)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # Best checkpoint 저장
        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_epoch = epoch + 1

            best_state = copy.deepcopy(
                model.state_dict()
            )

    # 가장 좋았던 epoch 복원
    model.load_state_dict(best_state)

    return (
        model,
        train_losses,
        val_losses,
        best_epoch,
        best_val_loss
    )

In [226]:
reset_seed(SEED)

model_exp11_best = MLPPlayerEmbedding(
    input_dim=X_train_final.shape[1],
    num_players=len(player_to_idx) + 1,
    player_embedding_dim=16
).to(device)

In [227]:
(
    model_exp11_best,
    train_loss_exp11_best,
    val_loss_exp11_best,
    best_epoch_exp11,
    best_val_exp11
) = train_embedding_model_best(
    model=model_exp11_best,
    train_loader=train_loader_exp11,
    val_loader=val_loader_exp11,
    mode="player",
    epochs=50,
    lr=0.001
)

In [228]:
result_exp11_best = evaluate_embedding_model(
    model_exp11_best,
    val_loader_exp11,
    mode="player"
)

print("[Exp11 Player Embedding - Best Checkpoint]")
print("Best Epoch:", best_epoch_exp11)

print(
    f"MAE  : {result_exp11_best['mae']:.4f}"
)

print(
    f"RMSE : {result_exp11_best['rmse']:.4f}"
)

print(
    f"R²   : {result_exp11_best['r2']:.4f}"
)

[Exp11 Player Embedding - Best Checkpoint]
Best Epoch: 1
MAE  : 2.4327
RMSE : 3.5288
R²   : 0.4494


In [229]:
exp11_best_preds = result_exp11_best["val_preds"]
exp11_best_targets = result_exp11_best["val_targets"]

print("[Exp11 Player Embedding - Best Checkpoint]")

evaluate_group(
    "Seen",
    exp11_best_targets,
    exp11_best_preds,
    seen_mask
)

evaluate_group(
    "Unseen",
    exp11_best_targets,
    exp11_best_preds,
    unseen_mask
)

[Exp11 Player Embedding - Best Checkpoint]
Seen       N=562 | MAE=2.5265 | RMSE=3.6348 | R²=0.4814
Unseen     N=207 | MAE=2.1782 | RMSE=3.2233 | R²=0.2604


In [230]:
reset_seed(SEED)

model_exp10_best = MLPTeamEmbeddingFinal(
    input_dim=X_train_final.shape[1],
    num_teams=len(team_to_idx) + 1,
    team_embedding_dim=8
).to(device)

(
    model_exp10_best,
    train_loss_exp10_best,
    val_loss_exp10_best,
    best_epoch_exp10,
    best_val_exp10
) = train_embedding_model_best(
    model=model_exp10_best,
    train_loader=train_loader_exp10,
    val_loader=val_loader_exp10,
    mode="team",
    epochs=50,
    lr=0.001
)

result_exp10_best = evaluate_embedding_model(
    model_exp10_best,
    val_loader_exp10,
    mode="team"
)

print("[Exp10 Team Embedding - Best Checkpoint]")
print("Best Epoch:", best_epoch_exp10)
print(f"MAE  : {result_exp10_best['mae']:.4f}")
print(f"RMSE : {result_exp10_best['rmse']:.4f}")
print(f"R²   : {result_exp10_best['r2']:.4f}")

[Exp10 Team Embedding - Best Checkpoint]
Best Epoch: 10
MAE  : 2.4705
RMSE : 3.5214
R²   : 0.4517


In [231]:
reset_seed(SEED)

model_exp12_best = MLPTeamPlayerEmbedding(
    input_dim=X_train_final.shape[1],
    num_teams=len(team_to_idx) + 1,
    num_players=len(player_to_idx) + 1,
    team_embedding_dim=8,
    player_embedding_dim=16
).to(device)

(
    model_exp12_best,
    train_loss_exp12_best,
    val_loss_exp12_best,
    best_epoch_exp12,
    best_val_exp12
) = train_embedding_model_best(
    model=model_exp12_best,
    train_loader=train_loader_exp12,
    val_loader=val_loader_exp12,
    mode="team_player",
    epochs=50,
    lr=0.001
)

result_exp12_best = evaluate_embedding_model(
    model_exp12_best,
    val_loader_exp12,
    mode="team_player"
)

print("[Exp12 Team + Player Embedding - Best Checkpoint]")
print("Best Epoch:", best_epoch_exp12)
print(f"MAE  : {result_exp12_best['mae']:.4f}")
print(f"RMSE : {result_exp12_best['rmse']:.4f}")
print(f"R²   : {result_exp12_best['r2']:.4f}")

[Exp12 Team + Player Embedding - Best Checkpoint]
Best Epoch: 1
MAE  : 2.4496
RMSE : 3.5330
R²   : 0.4481


In [232]:
embedding_best_compare = pd.DataFrame([
    {
        "experiment": "Exp9Cb_tabular",
        "mae": result_exp9cb["mae"],
        "rmse": result_exp9cb["rmse"],
        "r2": result_exp9cb["r2"],
        "best_epoch": None
    },
    {
        "experiment": "Exp10_team_embedding",
        "mae": result_exp10_best["mae"],
        "rmse": result_exp10_best["rmse"],
        "r2": result_exp10_best["r2"],
        "best_epoch": best_epoch_exp10
    },
    {
        "experiment": "Exp11_player_embedding",
        "mae": result_exp11_best["mae"],
        "rmse": result_exp11_best["rmse"],
        "r2": result_exp11_best["r2"],
        "best_epoch": best_epoch_exp11
    },
    {
        "experiment": "Exp12_team_player_embedding",
        "mae": result_exp12_best["mae"],
        "rmse": result_exp12_best["rmse"],
        "r2": result_exp12_best["r2"],
        "best_epoch": best_epoch_exp12
    }
])

embedding_best_compare

,experiment,mae,rmse,r2,best_epoch
0,Exp9Cb_tabular,2.441230,3.532324,0.448307,NaN
1,Exp10_team_embedding,2.470450,3.521419,0.451708,10.0
2,Exp11_player_embedding,2.432741,3.528795,0.449408,1.0
3,Exp12_team_player_embedding,2.449582,3.532955,0.448109,1.0


In [233]:
val_high_analysis = val_team[
    [
        "player",
        "team",
        "league",
        "position_group",
        "goals",
        "next_goals"
    ]
].copy()

val_high_analysis["pred_exp9cb"] = result_exp9cb["val_preds"]
val_high_analysis["pred_exp10_team"] = result_exp10_best["val_preds"]
val_high_analysis["pred_exp11_player"] = result_exp11_best["val_preds"]
val_high_analysis["pred_exp12_both"] = result_exp12_best["val_preds"]

val_high_analysis.head()

,player,team,league,position_group,goals,next_goals,pred_exp9cb,pred_exp10_team,pred_exp11_player,pred_exp12_both
0,Abde Ezzalzouli,Betis,La Liga,FW,1.0,2.0,2.818255,2.281266,1.641770,2.199831
1,Abdou Harroui,Frosinone,Serie A,FW,3.0,1.0,2.015152,3.341141,2.548152,2.736260
2,Abdoulaye Doucouré,Everton,Premier League,FW,7.0,3.0,3.008321,3.757860,3.329823,3.558905
3,Abdoulaye Touré,Le Havre,Ligue 1,MF,2.0,10.0,1.892533,1.390813,1.058996,1.020538
4,Abdón Prats,Mallorca,La Liga,FW,6.0,2.0,4.750561,4.531438,2.432927,3.003730


In [234]:
def evaluate_high_scorers(
    df,
    threshold,
    pred_cols
):
    subset = df[
        df["next_goals"] >= threshold
    ].copy()

    rows = []

    for model_name, pred_col in pred_cols.items():

        actual = subset["next_goals"]
        pred = subset[pred_col]

        rows.append({
            "threshold": f"{threshold}+",
            "model": model_name,
            "count": len(subset),
            "actual_mean": actual.mean(),
            "pred_mean": pred.mean(),
            "mae": mean_absolute_error(
                actual,
                pred
            ),
            "bias": (
                pred - actual
            ).mean()
        })

    return pd.DataFrame(rows)

In [235]:
pred_cols = {
    "Exp9Cb": "pred_exp9cb",
    "Exp10_Team": "pred_exp10_team",
    "Exp11_Player": "pred_exp11_player",
    "Exp12_Both": "pred_exp12_both"
}

In [236]:
high_10 = evaluate_high_scorers(
    val_high_analysis,
    10,
    pred_cols
)

high_15 = evaluate_high_scorers(
    val_high_analysis,
    15,
    pred_cols
)

high_20 = evaluate_high_scorers(
    val_high_analysis,
    20,
    pred_cols
)

high_score_compare = pd.concat(
    [
        high_10,
        high_15,
        high_20
    ],
    ignore_index=True
)

high_score_compare

,threshold,model,count,actual_mean,pred_mean,mae,bias
0,10+,Exp9Cb,87,14.436782,8.165205,6.824674,-6.271576
1,10+,Exp10_Team,87,14.436782,8.150716,6.696556,-6.286066
2,10+,Exp11_Player,87,14.436782,8.215303,6.548690,-6.221478
3,10+,Exp12_Both,87,14.436782,8.229236,6.540996,-6.207546
4,15+,Exp9Cb,32,19.718750,10.640303,9.535213,-9.078448
5,15+,Exp10_Team,32,19.718750,10.521133,9.360126,-9.197617
6,15+,Exp11_Player,32,19.718750,10.934229,8.836451,-8.784520
7,15+,Exp12_Both,32,19.718750,10.735554,9.019922,-8.983196
8,20+,Exp9Cb,16,23.125000,12.645255,11.272487,-10.479745
9,20+,Exp10_Team,16,23.125000,12.629718,10.820299,-10.495282


In [237]:
high_score_pivot = high_score_compare.pivot(
    index="model",
    columns="threshold",
    values=["mae", "pred_mean", "bias"]
)

high_score_pivot

mae                      pred_mean                        \
threshold          10+       15+        20+       10+        15+        20+   
model                                                                         
Exp10_Team    6.696556  9.360126  10.820299  8.150716  10.521133  12.629718   
Exp11_Player  6.548690  8.836451  10.394579  8.215303  10.934229  12.834283   
Exp12_Both    6.540996  9.019922  10.558039  8.229236  10.735554  12.640413   
Exp9Cb        6.824674  9.535213  11.272487  8.165205  10.640303  12.645255   

                  bias                       
threshold          10+       15+        20+  
model                                        
Exp10_Team   -6.286066 -9.197617 -10.495282  
Exp11_Player -6.221478 -8.784520 -10.290717  
Exp12_Both   -6.207546 -8.983196 -10.484586  
Exp9Cb       -6.271576 -9.078448 -10.479745

In [238]:
bins = [
    -0.1,
    0,
    4,
    9,
    14,
    19,
    float("inf")
]

labels = [
    "0",
    "1-4",
    "5-9",
    "10-14",
    "15-19",
    "20+"
]

val_high_analysis["goal_band"] = pd.cut(
    val_high_analysis["next_goals"],
    bins=bins,
    labels=labels
)

In [239]:
band_results = []

for band in labels:

    subset = val_high_analysis[
        val_high_analysis["goal_band"] == band
    ]

    if len(subset) == 0:
        continue

    for model_name, pred_col in pred_cols.items():

        actual = subset["next_goals"]
        pred = subset[pred_col]

        band_results.append({
            "goal_band": band,
            "model": model_name,
            "count": len(subset),
            "actual_mean": actual.mean(),
            "pred_mean": pred.mean(),
            "mae": mean_absolute_error(
                actual,
                pred
            ),
            "bias": (
                pred - actual
            ).mean()
        })

band_compare = pd.DataFrame(
    band_results
)

band_compare

,goal_band,model,count,actual_mean,pred_mean,mae,bias
0,0,Exp9Cb,185,0.000000,1.899350,1.904288,1.899350
1,0,Exp10_Team,185,0.000000,1.931639,1.931639,1.931639
2,0,Exp11_Player,185,0.000000,1.930854,1.930855,1.930855
3,0,Exp12_Both,185,0.000000,1.999992,1.999992,1.999992
4,1-4,Exp9Cb,355,2.140845,2.848612,1.600386,0.707767
5,1-4,Exp10_Team,355,2.140845,2.814922,1.625582,0.674077
6,1-4,Exp11_Player,355,2.140845,2.773689,1.543202,0.632844
7,1-4,Exp12_Both,355,2.140845,2.858495,1.543194,0.717650
8,5-9,Exp9Cb,142,6.591549,5.167706,2.557244,-1.423843
9,5-9,Exp10_Team,142,6.591549,5.191273,2.695358,-1.400276


In [240]:
band_mae_pivot = band_compare.pivot(
    index="goal_band",
    columns="model",
    values="mae"
)

band_mae_pivot

model,Exp10_Team,Exp11_Player,Exp12_Both,Exp9Cb
goal_band,,,,
0,1.931639,1.930855,1.999992,1.904288
1-4,1.625582,1.543202,1.543194,1.600386
10-14,5.146842,5.217628,5.098711,5.247633
15-19,7.899952,7.278324,7.481806,7.797939
20+,10.820299,10.394579,10.558039,11.272487
5-9,2.695358,2.788716,2.794572,2.557244


In [241]:
top_scorers = val_high_analysis[
    val_high_analysis["next_goals"] >= 20
].copy()

top_scorers = top_scorers[
    [
        "player",
        "team",
        "goals",
        "next_goals",
        "pred_exp9cb",
        "pred_exp10_team",
        "pred_exp11_player",
        "pred_exp12_both"
    ]
].sort_values(
    "next_goals",
    ascending=False
)

top_scorers

,player,team,goals,next_goals,pred_exp9cb,pred_exp10_team,pred_exp11_player,pred_exp12_both
404,Kylian Mbappé,Paris S-G,27.0,31.0,25.213972,22.862473,23.114529,21.944889
521,Mohamed Salah,Liverpool,18.0,29.0,15.798685,14.951023,14.505587,14.444208
628,Robert Lewandowski,Barcelona,19.0,27.0,18.095833,18.462994,17.722939,17.077471
272,Harry Kane,Bayern Munich,36.0,26.0,23.411425,23.103699,23.061827,22.886192
480,Mateo Retegui,Genoa,7.0,25.0,6.343607,6.099393,3.628719,3.673516
31,Alexander Isak,Newcastle Utd,21.0,23.0,14.299882,14.260765,15.434217,14.647949
570,Omar Marmoush,Eint Frankfurt,12.0,22.0,6.528037,7.696799,7.671137,7.842681
209,Erling Haaland,Manchester City,27.0,22.0,28.341936,24.600132,22.830893,22.587622
478,Mason Greenwood,Getafe,8.0,21.0,7.862408,7.345607,7.206501,7.868412
576,Ousmane Dembélé,Paris S-G,3.0,21.0,2.340825,3.449736,3.928492,4.169271
